In [85]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

np.random.seed(42)
random.seed(42)
os.makedirs("tradesphere", exist_ok=True)

countries      = ["India", "UAE", "USA", "UK", "Singapore"]
categories     = ["Electronics", "Clothing", "Food", "Books", "Home"]
statuses       = ["active", "inactive"]
order_statuses = ["pending", "shipped", "delivered", "cancelled", "returned"]
pay_methods    = ["credit_card", "debit_card", "UPI", "net_banking", "wallet"]
warehouses     = ["Mumbai", "Delhi", "Dubai", "London", "Singapore"]

def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end-start).days))

start_date = datetime(2022, 1, 1)
end_date   = datetime(2024, 12, 31)

first_names = ["Priya","Arjun","Sneha","Rahul","Divya","Karan",
               "Meera","Vikram","Anita","Suresh","Pooja","Amit",
               "Deepa","Rajesh","Nisha","Sanjay","Kavya","Rohan",
               "Sunita","Mohammed","Fatima","Ali","Sarah","John",
               "Emma","James","Aisha","Omar","Riya","Dev"]

last_names  = ["Mehta","Patel","Shah","Verma","Nair","Singh",
               "Iyer","Kumar","Sharma","Gupta","Khan","Ali",
               "Smith","Johnson","Williams","Brown","Jones",
               "Garcia","Martinez","Lee","Chen","Wang","Kim"]

# ── Customers ─────────────────────────────────────────────
customers = []
for i in range(1000):
    cid   = 1000 + i
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    name  = f"{fname} {lname}"
    if i % 50  == 0: name = "  " + name + "  "
    if i % 75  == 0: name = ""
    spent = round(random.uniform(500, 200000), 2)
    if i % 100 == 0: spent = None
    status = random.choices(statuses, weights=[80,20])[0]
    if i % 120 == 0:
        status = random.choice(["Active","ACTIVE","Invalid","unknown"])
    customers.append({
        "customer_id" : str(cid),
        "name"        : name,
        "email"       : f"{fname.lower()}.{lname.lower()}{i}@email.com",
        "country"     : random.choice(countries),
        "city"        : f"City_{i % 20}",
        "total_spent" : spent,
        "status"      : status,
        "join_date"   : random_date(start_date, end_date).strftime("%Y-%m-%d"),
        "is_premium"  : spent > 100000 if spent else False
    })
pd.DataFrame(customers).to_csv("tradesphere/customers.csv", index=False)

# ── Products ──────────────────────────────────────────────
products = []
for i in range(200):
    price    = round(random.uniform(100, 50000), 2)
    category = random.choice(categories)
    if i % 40 == 0: price    = None
    if i % 60 == 0: category = ""
    products.append({
        "product_id" : str(5000 + i),
        "name"       : f"Product_{i}",
        "category"   : category,
        "price"      : price,
        "stock_qty"  : random.randint(0, 500),
        "warehouse"  : random.choice(warehouses),
        "is_active"  : random.choice([True, True, True, False])
    })
pd.DataFrame(products).to_csv("tradesphere/products.csv", index=False)

# ── Orders ────────────────────────────────────────────────
customer_ids = [str(1000+i) for i in range(1000)]
product_ids  = [str(5000+i) for i in range(200)]
orders = []
for i in range(5000):
    status     = random.choices(order_statuses, weights=[10,20,50,15,5])[0]
    order_date = random_date(start_date, end_date)
    amount     = round(random.uniform(200, 80000), 2)
    if status in ["shipped","delivered"]:
        delivery_date = (order_date + timedelta(days=random.randint(1,14))).strftime("%Y-%m-%d")
    else:
        delivery_date = None
    if i % 80  == 0: amount = None
    if i % 150 == 0: status = "bad_status"
    orders.append({
        "order_id"       : str(10000+i),
        "customer_id"    : random.choice(customer_ids),
        "product_id"     : random.choice(product_ids),
        "amount"         : amount,
        "status"         : status,
        "payment_method" : random.choice(pay_methods),
        "order_date"     : order_date.strftime("%Y-%m-%d"),
        "delivery_date"  : delivery_date,
        "warehouse"      : random.choice(warehouses)
    })
pd.DataFrame(orders).to_csv("tradesphere/orders.csv", index=False)

# ── Payments ──────────────────────────────────────────────
payments = []
for order in orders:
    if random.random() > 0.05:
        pay_date = (
            datetime.strptime(order["order_date"], "%Y-%m-%d")
            + timedelta(days=random.randint(0,2))
        ).strftime("%Y-%m-%d")
        payments.append({
            "payment_id"   : f"PAY_{order['order_id']}",
            "order_id"     : order["order_id"],
            "customer_id"  : order["customer_id"],
            "amount"       : order["amount"],
            "method"       : order["payment_method"],
            "status"       : random.choices(
                                 ["success","failed","refunded"],
                                 weights=[90,5,5]
                             )[0],
            "payment_date" : pay_date
        })
pd.DataFrame(payments).to_csv("tradesphere/payments.csv", index=False)

# ── Confirm ───────────────────────────────────────────────
print("TradeSphere dataset ready")
for f in os.listdir("tradesphere"):
    df = pd.read_csv(f"tradesphere/{f}")
    print(f"  {f:25s} {len(df):>6,} rows  {len(df.columns)} cols")

TradeSphere dataset ready
  payments.csv               4,749 rows  7 cols
  customers.csv              1,000 rows  9 cols
  products.csv                 200 rows  7 cols
  orders.csv                 5,000 rows  9 cols


**Section 3.1 — Series and DataFrame**

**What this section does**
It introduces the two core data structures in Pandas. Everything in every other section — cleaning, filtering, groupby, merging — uses these two structures. If you do not understand them properly, every section after this will feel confusing.

**What it does to the data**
It takes data that exists as Python lists or dictionaries — or files on disk — and puts it into a structured table format that Pandas can operate on efficiently. Instead of looping through records one by one, you work with entire columns at once.

**What to expect from the output**
A Series looks like a single column with row numbers on the left. A DataFrame looks like a spreadsheet — rows, columns, headers, and row numbers.

**How to approach the code — the thinking**
Before writing any code, ask yourself two questions. What shape is my data — is it one column or multiple columns? What do I want to know about it — types, nulls, summary statistics?
Those two questions determine which commands you run.

**Block 1 — Creating a Series**

In [86]:
import pandas as pd

# A Series is one column of data
# Think of it as a single column from a spreadsheet
amounts = pd.Series([15000, 4000, 8000, 22000, 3000])
print(amounts)

0    15000
1     4000
2     8000
3    22000
4     3000
dtype: int64


**Block 2 — Operations on a Series happen to every value at once**

In [87]:
# No loop needed — Pandas applies this to every row
print(amounts * 1.18)   # add 18% tax to every amount
print(amounts > 10000)  # which amounts are high value
print(amounts.sum())    # total
print(amounts.mean())   # average

0    17700.0
1     4720.0
2     9440.0
3    25960.0
4     3540.0
dtype: float64
0     True
1    False
2    False
3     True
4    False
dtype: bool
52000
10400.0


**Block 3 — Creating a DataFrame**

In [88]:
# A DataFrame is multiple columns sharing the same index
# Think of it as a full spreadsheet table
customers = pd.DataFrame({
    "customer_id" : ["C001", "C002", "C003", "C004"],
    "name"        : ["Priya", "Arjun", "Sneha", "Rahul"],
    "spent"       : [45000, 4000, 12000, 72000],
    "status"      : ["active", "inactive", "active", "active"]
})

print(customers)

  customer_id   name  spent    status
0        C001  Priya  45000    active
1        C002  Arjun   4000  inactive
2        C003  Sneha  12000    active
3        C004  Rahul  72000    active


**Block 4 — The five commands you run on every new dataset**

In [89]:
# 1. Shape — how many rows and columns
print(customers.shape)

# 2. Data types — what type is each column
print(customers.dtypes)

# 3. Info — types plus null counts in one view
print(customers.info())

# 4. First few rows — quick sanity check
print(customers.head(3))

# 5. Statistical summary — min, max, mean, percentiles
print(customers.describe())

(4, 4)
customer_id    object
name           object
spent           int64
status         object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  4 non-null      object
 1   name         4 non-null      object
 2   spent        4 non-null      int64 
 3   status       4 non-null      object
dtypes: int64(1), object(3)
memory usage: 260.0+ bytes
None
  customer_id   name  spent    status
0        C001  Priya  45000    active
1        C002  Arjun   4000  inactive
2        C003  Sneha  12000    active
              spent
count      4.000000
mean   33250.000000
std    31340.867888
min     4000.000000
25%    10000.000000
50%    28500.000000
75%    51750.000000
max    72000.000000


**Block 5 — Null check — always run this**

In [90]:
# How many nulls in each column
print(customers.isnull().sum())

customer_id    0
name           0
spent          0
status         0
dtype: int64


In [91]:
# Load the orders file
orders = pd.read_csv("tradesphere/orders.csv")

# Your task — run all five commands:
# 1. Print the shape
print(orders.shape)
# 2. Print dtypes
print(orders.dtypes)
# 3. Print info()
print(orders.info())
# 4. Print head(5)
print(orders.head())
# 5. Print describe()
print(orders.describe)
# 6. Print isnull().sum()
print(orders.isnull().sum())

# Then answer these three questions
# from reading the output — not by writing code:

# Q1. How many rows and columns does orders have?
# 5000 Rows, 9 Columns
# Q2. Which column has the most null values?
# Delivery Date
# Q3. What is the maximum order amount?

(5000, 9)
order_id            int64
customer_id         int64
product_id          int64
amount            float64
status             object
payment_method     object
order_date         object
delivery_date      object
warehouse          object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        5000 non-null   int64  
 1   customer_id     5000 non-null   int64  
 2   product_id      5000 non-null   int64  
 3   amount          4937 non-null   float64
 4   status          5000 non-null   object 
 5   payment_method  5000 non-null   object 
 6   order_date      5000 non-null   object 
 7   delivery_date   3516 non-null   object 
 8   warehouse       5000 non-null   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 351.7+ KB
None
   order_id  customer_id  product_id    amount      status payment_metho

In [92]:
import pandas as pd

orders = pd.read_csv(
    "tradesphere/orders.csv",
    dtype       = {
        "order_id"    : str,
        "customer_id" : str,
        "product_id"  : str
    },
    parse_dates = ["order_date"]
)

print(orders.shape)
print("---")
print(orders.dtypes)
print("---")
print(orders.describe())
print("---")
print(orders.isnull().sum())

(5000, 9)
---
order_id                  object
customer_id               object
product_id                object
amount                   float64
status                    object
payment_method            object
order_date        datetime64[ns]
delivery_date             object
warehouse                 object
dtype: object
---
             amount                  order_date
count   4937.000000                        5000
mean   39787.453425  2023-06-29 09:12:40.320000
min      243.330000         2022-01-01 00:00:00
25%    19851.430000         2022-09-23 00:00:00
50%    38933.810000         2023-06-28 00:00:00
75%    60413.110000         2024-04-03 00:00:00
max    79942.600000         2024-12-31 00:00:00
std    23128.727590                         NaN
---
order_id             0
customer_id          0
product_id           0
amount              63
status               0
payment_method       0
order_date           0
delivery_date     1484
warehouse            0
dtype: int64


**Section 3.2 — Loading Data**

**What this section does**
Teaches you how to load data from different sources — CSV files, JSON files, and databases. And more importantly, how to load it correctly — with the right types, the right null handling, and the right date parsing from the start.

**What it does to the data**

It brings external data into your Pandas session as a DataFrame. The parameters you pass during loading directly determine the quality of what you get. Same file, different parameters — completely different results.

**What to expect from the output**

A DataFrame that is already partially clean before you even start cleaning. Correct types, correct nulls, correct dates — all handled at load time rather than after.

How to approach the code — the thinking
Before loading any file ask three questions:

**Which columns are IDs? Load them as str**

**Which columns are dates? Put them in parse_dates**

**Which values mean null? Put them in na_values**

Answer those three questions first. Then write the read_csv.

**Block 1 — Bad loading vs good loading**

In [93]:
# Bad loading — no parameters
orders_bad = pd.read_csv("tradesphere/orders.csv")
print(orders_bad.dtypes)
print(orders_bad["order_id"].dtype)

order_id            int64
customer_id         int64
product_id          int64
amount            float64
status             object
payment_method     object
order_date         object
delivery_date      object
warehouse          object
dtype: object
int64


In [94]:
# Good loading — with proper parameters
order_good = pd.read_csv(
    "tradesphere/orders.csv",
    dtype = {
        "order_id" : str,
        "customer_id" : str,
        "product_id" : str
    },
    parse_dates = ["order_date"],
    na_values = ["", " ", "bad_status", "NULL", "N/A"]
    )
print(order_good.dtypes)

order_id                  object
customer_id               object
product_id                object
amount                   float64
status                    object
payment_method            object
order_date        datetime64[ns]
delivery_date             object
warehouse                 object
dtype: object


**Block 2 — Loading customers with all parameters**

In [95]:
customers = pd.read_csv(
    "tradesphere/customers.csv",
    dtype = {
        "customer_id" : str,
        "country"     : "category",
        "status"      : "category"
    },
    parse_dates = ["join_date"],
    na_values   = ["", " ", "NULL"]
)

print(customers.dtypes)
print("---")
print(customers.memory_usage(deep=True))

customer_id            object
name                   object
email                  object
country              category
city                   object
total_spent           float64
status               category
join_date      datetime64[ns]
is_premium               bool
dtype: object
---
Index            132
customer_id    53000
name           59557
email          72783
country         1439
city           55500
total_spent     8000
status          1450
join_date       8000
is_premium      1000
dtype: int64


**Block 4 — Chunked reading for large files**

This is how you handle files too large to fit in memory:

In [96]:
chunk_list = []

for chunk in pd.read_csv(
    "tradesphere/orders.csv",
    chunksize = 1000,
    dtype     = {"order_id": str, "customer_id": str}
):
    # Process each chunk — here we just filter delivered orders
    delivered_chunk = chunk[chunk["status"] == "delivered"]
    chunk_list.append(delivered_chunk)

# Combine all chunks
delivered_orders = pd.concat(chunk_list, ignore_index=True)
print(f"Total delivered orders: {len(delivered_orders)}")

Total delivered orders: 2530


In [97]:
# Task 1
# Load customers.csv with these requirements:
# - customer_id as string
# - country as category
# - status as category
# - join_date parsed as datetime
# - empty strings treated as null
# After loading print dtypes and isnull().sum()
# What dtype did country get?

# Task 2
# Load orders.csv in chunks of 500 rows
# While reading each chunk — count only orders
# where amount > 20000
# Print the final count of high value orders
# Hint — use a counter variable, add to it each chunk

# Task 3
# Load products.csv normally
# Print shape and dtypes
# Which column has the most nulls?

In [98]:
#Task 1
customers = pd.read_csv(
    "tradesphere/customers.csv",
    dtype = {
        "customer_id" : str,
        "country"     : "category",
        "status"      : "category"
    },
    parse_dates = ["join_date"],
    na_values   = ["", " ", "NULL"]
)

print(customers.dtypes)
print(customers.isnull().sum())

customer_id            object
name                   object
email                  object
country              category
city                   object
total_spent           float64
status               category
join_date      datetime64[ns]
is_premium               bool
dtype: object
customer_id     0
name           14
email           0
country         0
city            0
total_spent    10
status          0
join_date       0
is_premium      0
dtype: int64


In [99]:
#Task 2:
high_value_count = 0

for chunk in pd.read_csv(
    "tradesphere/orders.csv",
    chunksize = 500,
    dtype     = {"order_id": str, "customer_id": str}
):
    high_value_count += len(chunk[chunk["amount"] > 20000])

print(f"High value orders: {high_value_count}")

High value orders: 3694


In [100]:
# Task 3
# Load products.csv normally
# Print shape and dtypes
# Which column has the most nulls?
temp = pd.read_csv("tradesphere/products.csv", nrows=2)
print(temp.columns.tolist())

products = pd.read_csv(
    "tradesphere/products.csv",
    dtype = {
        "product_id" : str,
        "category"   : "category",
        "warehouse"  : "category"
    }
)

print(products.shape)
print("---")
print(products.dtypes)
print("---")
print(products.isnull().sum())

['product_id', 'name', 'category', 'price', 'stock_qty', 'warehouse', 'is_active']
(200, 7)
---
product_id      object
name            object
category      category
price          float64
stock_qty        int64
warehouse     category
is_active         bool
dtype: object
---
product_id    0
name          0
category      4
price         5
stock_qty     0
warehouse     0
is_active     0
dtype: int64


Section 3.3 — Selecting and Filtering

**What this section does**

Teaches you how to pull specific rows and columns out of a DataFrame. Every pipeline filters data — keep only active customers, keep only delivered orders, keep only records from 2024. This is how you do that.

**What it does to the data**

Nothing permanent. Selecting and filtering creates a new smaller DataFrame from the original. The original stays untouched unless you explicitly overwrite it.

**What to expect from the output**

A DataFrame with fewer rows, fewer columns, or both — containing only what you asked for.

**How to approach the code — the thinking**

Before writing any filter ask one question — what is my condition?

Write it in plain English first.
"I want orders where status is delivered"

"I want customers from India who are active"

"I want orders where amount is above 10000"

Full worked example

**Make sure all three DataFrames are loaded first:**

In [101]:
import pandas as pd

customers = pd.read_csv(
    "tradesphere/customers.csv",
    dtype       = {"customer_id": str,
                   "country": "category",
                   "status": "category"},
    parse_dates = ["join_date"],
    na_values   = ["", " ", "NULL"]
)

orders = pd.read_csv(
    "tradesphere/orders.csv",
    dtype       = {"order_id": str,
                   "customer_id": str,
                   "product_id": str},
    parse_dates = ["order_date"]
)

products = pd.read_csv(
    "tradesphere/products.csv",
    dtype = {"product_id": str,
             "category": "category",
             "warehouse": "category"}
)

**Block 1 — Selecting columns**

In [102]:
# One column — returns a Series
print(orders["status"])

# Multiple columns — returns a DataFrame
print(orders[["order_id", "amount", "status"]].head())

0       bad_status
1         returned
2          shipped
3         returned
4          shipped
           ...    
4995     delivered
4996     cancelled
4997       shipped
4998     cancelled
4999     delivered
Name: status, Length: 5000, dtype: object
  order_id    amount      status
0    10000       NaN  bad_status
1    10001  13068.13    returned
2    10002  29699.84     shipped
3    10003  29958.61    returned
4    10004  32484.07     shipped


**Block 2 — loc and iloc**

In [103]:
# loc — select by label name
print(orders.loc[0:4, ["order_id", "amount"]])

# iloc — select by position number
print(orders.iloc[0:5, 0:3])

#loc uses column names and row labels.
#iloc uses integer positions.
#In production use loc by default — it is explicit and readable.
#Use iloc only when you need positional access like "give me the last 5 rows".


  order_id    amount
0    10000       NaN
1    10001  13068.13
2    10002  29699.84
3    10003  29958.61
4    10004  32484.07
  order_id customer_id product_id
0    10000        1728       5137
1    10001        1119       5010
2    10002        1575       5181
3    10003        1552       5041
4    10004        1786       5037


**Block 3 — Boolean filtering — the most important pattern**

In [104]:
# Step 1 — write the condition in English
# "I want only delivered orders"

# Step 2 — translate to Pandas
delivered = orders[orders["status"] == "delivered"]
print(f"Delivered orders: {len(delivered)}")

# Step 3 — verify
print(delivered["status"].unique())

Delivered orders: 2530
['delivered']


In [105]:
# AND — both must be true — use &
# Parentheses around each condition are mandatory
high_value_delivered = orders[
    (orders["status"] == "delivered") &
    (orders["amount"] > 20000)
]
print(f"High value delivered: {len(high_value_delivered)}")

# OR — either can be true — use |
india_or_uae = customers[
    (customers["country"] == "India") |
    (customers["country"] == "UAE")
]
print(f"India or UAE customers: {len(india_or_uae)}")

# NOT — exclude — use ~
not_cancelled = orders[
    orders["status"] != "cancelled"
]
print(f"Non-cancelled: {len(not_cancelled)}")

High value delivered: 1880
India or UAE customers: 374
Non-cancelled: 4265


**Block 5 — isin — filtering against a list**


In [106]:
# Instead of writing multiple OR conditions
target_countries = ["India", "UAE", "Singapore"]

target_customers = customers[
    customers["country"].isin(target_countries)
]
print(f"Target customers: {len(target_customers)}")

# Exclude a list — use ~ to flip
other_customers = customers[
    ~customers["country"].isin(target_countries)
]
print(f"Other customers: {len(other_customers)}")

Target customers: 580
Other customers: 420


**Block 6 — query() — readable filtering**

In [107]:
# Same result as boolean filter but reads like English
result = orders.query(
    "amount > 20000 and status == 'delivered'"
)
print(f"High value delivered via query: {len(result)}")

High value delivered via query: 1880


**Block 7 — Filtering nulls**

In [108]:
# Rows where delivery_date is null
no_delivery = orders[orders["delivery_date"].isna()]
print(f"No delivery date: {len(no_delivery)}")

# Rows where delivery_date exists
has_delivery = orders[orders["delivery_date"].notna()]
print(f"Has delivery date: {len(has_delivery)}")

# Empty string is NOT null — different check
empty_names = customers[customers["name"] == ""]
print(f"Empty names: {len(empty_names)}")

No delivery date: 1484
Has delivery date: 3516
Empty names: 0


**Practice tasks**

**Three tasks on TradeSphere data:**

In [109]:
# Task 1
# From orders — find all orders that are:
# - status is "shipped" OR "pending"
# - amount is greater than 5000
# Print how many rows match
# Print the unique statuses in the result
#   to confirm only shipped and pending appear

shipped_pending = orders[
    (orders["status"].isin(["shipped", "pending"])) &
    (orders["amount"] > 5000)
]
print(f"Shipped and Pending Order: {len(shipped_pending)}")
print(shipped_pending["status"].unique())

Shipped and Pending Order: 1356
['shipped' 'pending']


In [110]:
# Task 2
# From customers — find customers who:
# - joined after 2023-01-01
# - are from India
# - have total_spent above 10000
# Hint — join_date is already datetime
# Use join_date > "2023-01-01" in your condition
# Print how many customers match

# AND — both must be true — use &
# Parentheses around each condition are mandatory
find_customer = customers[
    (customers["join_date"] > "2023-01-01") &
    (customers["country"] == "India")&
    (customers["total_spent"] > 10000)
]
print(f"High value Find_customer: {len(find_customer)}")




High value Find_customer: 117


In [111]:

#Task 3
# From products — find products where:
# - category is in ["Electronics", "Clothing", "Books"]
# - price is NOT null
# - is_active is True
# Print how many products match
# Print the category value_counts of the result

# Instead of writing multiple OR conditions
target_products = ["Electronics", "Clothing", "Books"]

target_pro = products[
    (products["category"].isin(target_products)) &
    (products["price"].notna()) &
    (products["is_active"] == True)
]
# See all matching rows
print(target_pro)

# See first 10 rows
print(target_pro.head(10))

# See specific columns only
print(target_pro[["product_id", "name", "category", "price"]])

# See summary — how many per category
print(target_pro["category"].value_counts())

# See one specific row by position
print(target_pro.iloc[0])
print(f"Matching products: {len(target_pro)}")
print(target_pro["category"].value_counts())

    product_id         name     category     price  stock_qty  warehouse  \
2         5002    Product_2        Books   7853.78        456  Singapore   
3         5003    Product_3        Books  23177.26        475     Mumbai   
8         5008    Product_8  Electronics   9450.21        442      Dubai   
11        5011   Product_11     Clothing  26302.51         57      Dubai   
12        5012   Product_12  Electronics  38911.74        227  Singapore   
..         ...          ...          ...       ...        ...        ...   
191       5191  Product_191     Clothing   7609.85        266     Mumbai   
193       5193  Product_193        Books  17400.34        377     Mumbai   
194       5194  Product_194  Electronics   9534.13        446      Delhi   
198       5198  Product_198     Clothing  39484.46        247      Dubai   
199       5199  Product_199  Electronics  38510.51         75      Dubai   

     is_active  
2         True  
3         True  
8         True  
11        True  
12

**Section 3.4 — Indexing and MultiIndex**

**What this section does**

Teaches you how to use the DataFrame index properly — setting meaningful labels instead of default numbers, and understanding the two-level index that groupby operations produce.

**What it does to the data**

Changes which column acts as the row label. Does not add or remove data — just reorganises how rows are addressed.

**What to expect from the output**

After set_index() — row numbers on the left are replaced by meaningful values like customer IDs or product names. After groupby — a two-level index appears which needs to be flattened with reset_index().

Full worked example

**Block 1 — Default index vs meaningful index**

In [112]:
# Default — just numbers
print(products.head(3))
print("Index:", products.index.tolist()[:3])

  product_id       name     category     price  stock_qty  warehouse  \
0       5000  Product_0          NaN       NaN        169      Dubai   
1       5001  Product_1  Electronics  45816.54        330      Delhi   
2       5002  Product_2        Books   7853.78        456  Singapore   

   is_active  
0      False  
1      False  
2       True  
Index: [0, 1, 2]


In [113]:
# Set product_id as index
products_indexed = products.set_index("product_id")
print(products_indexed.head(3))
print("Index:", products_indexed.index.tolist()[:3])

                 name     category     price  stock_qty  warehouse  is_active
product_id                                                                   
5000        Product_0          NaN       NaN        169      Dubai      False
5001        Product_1  Electronics  45816.54        330      Delhi      False
5002        Product_2        Books   7853.78        456  Singapore       True
Index: ['5000', '5001', '5002']


**Block 2 — Lookup using loc after set_index**

In [114]:
# Look up one specific product
print(products_indexed.loc["5005"])

# Look up multiple specific products
print(products_indexed.loc[["5001", "5010", "5025"]])

name         Product_5
category          Food
price          10492.3
stock_qty          409
warehouse        Dubai
is_active         True
Name: 5005, dtype: object
                  name     category     price  stock_qty warehouse  is_active
product_id                                                                   
5001         Product_1  Electronics  45816.54        330     Delhi      False
5010        Product_10         Food  43864.73        231     Dubai       True
5025        Product_25         Home   3655.02        456     Dubai      False


**Block 3 — reset_index — putting it back**

In [115]:
# After groupby the result has the groupby column as index
status_counts = orders.groupby("status")["amount"].sum()
print(status_counts)
print(type(status_counts))
print("Index:", status_counts.index.tolist())

status
bad_status     1275103.08
cancelled     28977198.62
delivered     99808806.52
pending       19384937.24
returned       9608935.49
shipped       37375676.61
Name: amount, dtype: float64
<class 'pandas.core.series.Series'>
Index: ['bad_status', 'cancelled', 'delivered', 'pending', 'returned', 'shipped']


In [116]:
# Reset brings it back as a regular column
status_counts = status_counts.reset_index()
print(status_counts)
print(type(status_counts))

       status       amount
0  bad_status   1275103.08
1   cancelled  28977198.62
2   delivered  99808806.52
3     pending  19384937.24
4    returned   9608935.49
5     shipped  37375676.61
<class 'pandas.core.frame.DataFrame'>


**Block 4 — MultiIndex — what it is**

In [117]:
revenue = (
    orders.groupby(["status", "warehouse"])["amount"]
    .sum()
    .reset_index()
)

revenue.columns = ["status", "warehouse", "total_revenue"]
revenue = revenue.sort_values("total_revenue", ascending=False)
print(revenue.head(10))

       status  warehouse  total_revenue
10  delivered      Delhi    21795221.24
12  delivered     London    20203820.71
14  delivered  Singapore    19527322.33
11  delivered      Dubai    19260716.85
13  delivered     Mumbai    19021725.39
27    shipped     London     8027431.40
29    shipped  Singapore     7997295.69
28    shipped     Mumbai     7861766.63
26    shipped      Dubai     6883872.12
7   cancelled     London     6609697.20


**Block 5 — sort_values**

In [118]:
# Sort by one column
print(orders.sort_values("amount", ascending=False).head(5))

# Sort by multiple columns
print(orders.sort_values(
    ["status", "amount"],
    ascending=[True, False]
).head(10))

     order_id customer_id product_id    amount     status payment_method  \
1375    11375        1348       5105  79942.60  delivered    credit_card   
886     10886        1584       5187  79913.01    shipped    net_banking   
1706    11706        1209       5182  79886.60  delivered    net_banking   
4704    14704        1257       5196  79875.26  delivered            UPI   
4026    14026        1074       5149  79873.90    pending            UPI   

     order_date delivery_date  warehouse  
1375 2024-02-17    2024-02-22  Singapore  
886  2024-10-27    2024-11-06     Mumbai  
1706 2022-12-01    2022-12-15     Mumbai  
4704 2023-08-24    2023-08-31      Delhi  
4026 2023-06-11           NaN      Delhi  
     order_id customer_id product_id    amount      status payment_method  \
4200    14200        1355       5184  78486.33  bad_status            UPI   
1050    11050        1651       5158  76465.79  bad_status    net_banking   
3000    13000        1220       5071  75821.90  bad_st

**Practice tasks**

In [119]:
# Task 1
# Set order_id as the index of orders
# Look up order "10042" using loc
# Print the result

orders_indexed = orders.set_index("order_id")
print(orders_indexed.loc["10042"])
print(orders_indexed.head(3))


customer_id                      1219
product_id                       5131
amount                       78507.83
status                        shipped
payment_method             debit_card
order_date        2023-12-28 00:00:00
delivery_date              2024-01-04
warehouse                      London
Name: 10042, dtype: object
         customer_id product_id    amount      status payment_method  \
order_id                                                               
10000           1728       5137       NaN  bad_status            UPI   
10001           1119       5010  13068.13    returned            UPI   
10002           1575       5181  29699.84     shipped    credit_card   

         order_date delivery_date  warehouse  
order_id                                      
10000    2022-12-07    2022-12-10      Delhi  
10001    2022-11-11           NaN  Singapore  
10002    2023-06-18    2023-06-22      Dubai  


In [120]:
# Task 2
# Group orders by warehouse and payment_method
# Sum the amount for each combination
# Reset index and rename columns cleanly
# Sort by total amount descending
# Print top 5 rows


order_method = (
    orders.groupby(["warehouse", "payment_method"])["amount"]
    .sum()
    .reset_index()
)

order_method.columns = ["warehouse", "payment_method", "total_revenue"]
order_method = order_method.sort_values("total_revenue", ascending=False)
print(order_method.head(5))

    warehouse payment_method  total_revenue
11     London    credit_card     8834968.02
2       Delhi     debit_card     8754832.01
1       Delhi    credit_card     8689198.00
20  Singapore            UPI     8469743.67
22  Singapore     debit_card     8357263.70


In [121]:
# Task 3
# Sort the products DataFrame by price descending
# Print the top 3 most expensive products
# Show only name, category, price columns
abc = products[["name", "category", "price"]]

print(abc.sort_values("price", ascending=False).head(3))

            name  category     price
169  Product_169      Home  49564.55
82    Product_82  Clothing  49205.18
7      Product_7      Home  49182.58


**Section 3.5 — Data Cleaning**

**What this section does**

Teaches you how to fix broken data — nulls, wrong types, inconsistent strings, duplicates. This is 60% of real DE work. Every dataset you ever load will have these problems.

**What it does to the data**

Actually changes the data — removes bad rows, fills missing values, standardises inconsistent text, corrects wrong types. Unlike filtering which just views data, cleaning permanently modifies it.

**What to expect from the output**

A DataFrame with fewer nulls, consistent values, correct types, and no duplicates. Smaller than the original but trustworthy.


The cleaning sequence — memorise this
Always follow this order. Never skip steps:
1. Audit    — understand what is broken
2. Normalise — fix text — strip, lowercase
3. Filter   — remove invalid values
4. Fill     — handle remaining nulls
5. Dedupe   — remove duplicates
6. Types    — convert to correct dtypes
7. Verify   — check everything worked

Full worked example

**Block 1 — Audit first — never skip this**

In [122]:
print("=== NULL COUNTS ===")
print(customers.isnull().sum())

print("\n=== STATUS VALUES ===")
print(customers["status"].value_counts())

print("\n=== COUNTRY VALUES ===")
print(customers["country"].value_counts())

print("\n=== SHAPE ===")
print(customers.shape)

=== NULL COUNTS ===
customer_id     0
name           14
email           0
country         0
city            0
total_spent    10
status          0
join_date       0
is_premium      0
dtype: int64

=== STATUS VALUES ===
status
active      824
inactive    167
ACTIVE        4
Active        4
Invalid       1
Name: count, dtype: int64

=== COUNTRY VALUES ===
country
USA          214
UK           206
Singapore    206
UAE          194
India        180
Name: count, dtype: int64

=== SHAPE ===
(1000, 9)


**Block 2 — Normalise text first**

In [123]:
# Always normalise BEFORE filtering
# "Active" must become "active" before you can filter it out
customers["status"] = (
    customers["status"]
    .str.strip()
    .str.lower()
)

print(customers["status"].value_counts())

status
active      832
inactive    167
invalid       1
Name: count, dtype: int64


**Block 3 — Filter invalid values**

In [124]:
valid_statuses = {"active", "inactive"}

# Check what is invalid
invalid = customers[~customers["status"].isin(valid_statuses)]
print(f"Invalid status rows: {len(invalid)}")
print(invalid["status"].value_counts())

Invalid status rows: 1
status
invalid    1
Name: count, dtype: int64


In [125]:
customers_clean = customers[
    customers["status"].isin(valid_statuses)
].copy()

print(f"\nBefore: {len(customers)}")
print(f"After:  {len(customers_clean)}")


Before: 1000
After:  999


**Block 4 — Handle nulls — three strategies**

In [126]:
# Strategy 1 — Drop rows where critical field is null
# name is required — a customer without a name cannot be identified
print(f"Before name drop: {len(customers_clean)}")
customers_clean = customers_clean.dropna(subset=["name"])
print(f"After name drop:  {len(customers_clean)}")

Before name drop: 999
After name drop:  986


In [127]:
# Strategy 2 — Fill with a default value
# Missing total_spent means zero — customer registered but never bought
customers_clean["total_spent"] = (
    customers_clean["total_spent"].fillna(0)
)

# Strategy 3 — Fill with group mean
# Missing product price filled with category average
products["price"] = products.groupby("category")["price"].transform(
    lambda x: x.fillna(x.mean())
)

print("\nNull counts after filling:")
print(customers_clean.isnull().sum())


Null counts after filling:
customer_id    0
name           0
email          0
country        0
city           0
total_spent    0
status         0
join_date      0
is_premium     0
dtype: int64


/tmp/ipykernel_6994/1744089483.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  products["price"] = products.groupby("category")["price"].transform(


**Block 5 — Fix empty strings**


In [128]:
# Empty string "" is NOT null — isna() will not catch it
empty_names = customers_clean[customers_clean["name"] == ""]
print(f"Empty name strings: {len(empty_names)}")

# Convert empty strings to NaN then drop
customers_clean["name"] = customers_clean["name"].replace("", pd.NA)
customers_clean = customers_clean.dropna(subset=["name"])

print(f"After empty string fix: {len(customers_clean)}")

Empty name strings: 0
After empty string fix: 986


**Block 6 — Deduplication**

In [129]:
print(f"Before dedupe: {len(customers_clean)}")
print(f"Unique IDs:    {customers_clean['customer_id'].nunique()}")

customers_clean = customers_clean.drop_duplicates(
    subset = ["customer_id"],
    keep   = "first"
)

print(f"After dedupe:  {len(customers_clean)}")

Before dedupe: 986
Unique IDs:    986
After dedupe:  986


**Block 7 — Verify — always the last step**

In [130]:
print("=== FINAL AUDIT ===")
print(f"Shape: {customers_clean.shape}")
print("\nNull counts:")
print(customers_clean.isnull().sum())
print("\nStatus values:")
print(customers_clean["status"].value_counts())
print("\nDtype check:")
print(customers_clean.dtypes)

=== FINAL AUDIT ===
Shape: (986, 9)

Null counts:
customer_id    0
name           0
email          0
country        0
city           0
total_spent    0
status         0
join_date      0
is_premium     0
dtype: int64

Status values:
status
active      820
inactive    166
Name: count, dtype: int64

Dtype check:
customer_id            object
name                   object
email                  object
country              category
city                   object
total_spent           float64
status                 object
join_date      datetime64[ns]
is_premium               bool
dtype: object


**Practice tasks**


**Task 1** — Clean orders DataFrame
Input  — orders DataFrame (raw, messy)
Goal   — produce orders_clean (trusted, usable)

What to fix:
- status has inconsistent casing — Active, ACTIVE, bad_status
- status has invalid values — bad_status
- amount has null values

Steps:
1. Print status value_counts before cleaning
2. Normalise status — strip whitespace, lowercase
3. Keep only valid statuses:
   pending, shipped, delivered, cancelled, returned
4. Fill null amounts with median amount
5. Print shape before and after
6. Print status value_counts after
7. Print isnull().sum() to confirm no nulls remain

In [131]:
# Task 1 — Clean the orders DataFrame
# Follow the cleaning sequence:
# Step 1 — print status value_counts before cleaning
# Step 2 — normalise status — strip and lowercase
# Step 3 — keep only valid statuses:
#           pending, shipped, delivered, cancelled, returned
# Step 4 — fill null amounts with the median amount
# Step 5 — print shape before and after
# Step 6 — print status value_counts after cleaning
# Step 7 — print isnull().sum() to confirm no null amounts


#Step 1 — print status value_counts before cleaning
print("STEP 1=== BEFORE CLEANING ===")
print(orders["status"].value_counts())
print(orders.isnull().sum())

#Step 2 — normalise status — strip and lowercase
print("Step 2")
orders["status"] = orders["status"].str.strip().str.lower()
print(orders['status'])

#Step 3 — keep only valid statuses:
#           pending, shipped, delivered, cancelled, returned
print("Step 3")
valid_status = {"pending", "shipped", "delivered",
                "cancelled", "returned"}

orders_clean = orders[
    orders["status"].isin(valid_status)
].copy()
print(orders['status'])

#Step 4 — fill null amounts with the median amount
print("Step 4")
orders_clean["amount"] = orders_clean["amount"].fillna(
    orders_clean["amount"].median()
)
print(orders_clean)

#Step 5 — print shape before and after
print("Step 5")
print(f"\nBefore: {len(orders)}")
print(f"After:  {len(orders_clean)}")

#Step 6 — print status value_counts after cleaning
print("Step 6")
print("\n=== AFTER ===")
print(orders_clean["status"].value_counts())

#Step 7 — print isnull().sum() to confirm no null amounts
print("Step 7")
print(orders_clean.isnull().sum())



STEP 1=== BEFORE CLEANING ===
status
delivered     2530
shipped        960
cancelled      735
pending        496
returned       245
bad_status      34
Name: count, dtype: int64
order_id             0
customer_id          0
product_id           0
amount              63
status               0
payment_method       0
order_date           0
delivery_date     1484
warehouse            0
dtype: int64
Step 2
0       bad_status
1         returned
2          shipped
3         returned
4          shipped
           ...    
4995     delivered
4996     cancelled
4997       shipped
4998     cancelled
4999     delivered
Name: status, Length: 5000, dtype: object
Step 3
0       bad_status
1         returned
2          shipped
3         returned
4          shipped
           ...    
4995     delivered
4996     cancelled
4997       shipped
4998     cancelled
4999     delivered
Name: status, Length: 5000, dtype: object
Step 4
     order_id customer_id product_id    amount     status payment_method  \
1   

**Task 2** — Clean products DataFrame
Input  — products DataFrame (raw, messy)
Goal   — produce products_clean (trusted, usable)

What to fix:
- category has inconsistent casing and empty strings
- price has null values — fill with category average
  (Electronics nulls → Electronics mean price)
  (Clothing nulls → Clothing mean price)

Steps:
1. Normalise category — strip and lowercase
2. Remove rows where category is empty string
3. Fill null prices with mean price of that category
   Use groupby + transform + lambda
4. Print shape before and after
5. Print isnull().sum() to confirm

In [132]:
# Step 1 — normalise category
products["category"] = (
    products["category"]
    .str.strip()
    .str.lower()
)

# Step 2 — remove empty category rows
products["category"] = products["category"].replace("", pd.NA)
products_clean = products.dropna(subset=["category"]).copy()

# Step 3 — fill null prices with category mean
products_clean["price"] = (
    products_clean
    .groupby("category")["price"]
    .transform(lambda x: x.fillna(x.mean()))
)

# Step 4 — shape before and after
print(f"Before: {len(products)}")
print(f"After:  {len(products_clean)}")

# Step 5 — null check
print(products_clean.isnull().sum())

Before: 200
After:  196
product_id    0
name          0
category      0
price         0
stock_qty     0
warehouse     0
is_active     0
dtype: int64


**Task 3** — Answer business questions on cleaned orders
Input  — orders_clean from Task 1

Question 1:
How many orders have amount above 40000?

Question 2:
Which status has the highest average amount?
Steps — groupby status, calculate mean,
        sort descending, print top result

In [133]:
# Task 3 answers
high = orders_clean[orders_clean["amount"] > 40000]
print(f"\nOrders above 40000: {len(high)}")

avg_by_status = (
    orders_clean
    .groupby("status")["amount"]
    .mean()
    .reset_index()
    .sort_values("amount", ascending=False)
)
avg_by_status.columns = ["status", "avg_amount"]
avg_by_status["avg_amount"] = avg_by_status["avg_amount"].round(2)
print(avg_by_status)


Orders above 40000: 2378
      status  avg_amount
3   returned    40014.07
0  cancelled    40006.97
1  delivered    39834.53
2    pending    39553.13
4    shipped    39378.75


**Section 3.6 — Datetime Handling**

**What this section does**

Teaches you how to work with date and time data. Every real business question involves time — revenue this month, orders last quarter, customers inactive for 90 days. None of that is possible without proper datetime handling.

**What it does to the data**

Converts date strings into proper datetime objects that Pandas understands. Then extracts parts — year, month, day — or calculates time differences between dates.

**What to expect from the output**

New columns containing year numbers, month numbers, day names. Time differences in days. Filtered DataFrames for specific date ranges.

**Block 1 — Check current state of dates**

In [134]:
print(orders_clean["order_date"].dtype)
print(orders_clean["delivery_date"].dtype)

datetime64[ns]
object


**Block 2 — Convert delivery_date to datetime**

In [135]:
orders_clean['delivery_date'] = pd.to_datetime(
    orders_clean['delivery_date'],
    errors= "coerce"
)

print(orders_clean['delivery_date'].dtype)
print(orders_clean['delivery_date'].isnull().sum())

datetime64[ns]
1476


**Block 3 — Extract date parts**

In [136]:
orders_clean['year'] = orders_clean['delivery_date'].dt.year
orders_clean['month'] = orders_clean['delivery_date'].dt.month
orders_clean['weekday'] = orders_clean['delivery_date'].dt.day_name()
orders_clean['quarter'] = orders_clean['delivery_date'].dt.quarter


print(orders_clean[[
    "order_id", "order_date",
    "year", "month", "weekday", "quarter"
]].head(5))

  order_id order_date    year  month   weekday  quarter
1    10001 2022-11-11     NaN    NaN       NaN      NaN
2    10002 2023-06-18  2023.0    6.0  Thursday      2.0
3    10003 2022-10-30     NaN    NaN       NaN      NaN
4    10004 2022-02-21  2022.0    3.0   Tuesday      1.0
5    10005 2024-11-05  2024.0   11.0    Sunday      4.0


**Block 4 — Time based filtering**


In [137]:
# Orders from 2024 only
orders_2024 = orders_clean[
    orders_clean["order_date"].dt.year == 2024
]
print(f"2024 orders: {len(orders_2024)}")


# Orders between two dates
start = pd.Timestamp("2024-01-01")
end   = pd.Timestamp("2024-06-30")

orders_h1 = orders_clean[
    orders_clean["order_date"].between(start, end)
]
print(f"H1 2024 orders: {len(orders_h1)}")

2024 orders: 1661
H1 2024 orders: 847


**Block 5 — Date arithmetic**

In [138]:
# Calculate delivery time in days

delivered = orders_clean[
    orders_clean['status'] == "delivered"
].copy()

delivered["delivery_days"] = (
    delivered["delivery_date"] - delivered["order_date"]
).dt.days

print(f"Average delivery days: {delivered['delivery_days'].mean():.1f}")
print(f"Max delivery days    : {delivered['delivery_days'].max()}")
print(f"Min delivery days    : {delivered['delivery_days'].min()}")

# Flag late deliveries — more than 10 days
delivered["is_late"] = delivered["delivery_days"] > 10
print(f"\nLate deliveries: {delivered['is_late'].sum()}")
print(f"Late percentage: {delivered['is_late'].mean()*100:.1f}%")

Average delivery days: 7.4
Max delivery days    : 14
Min delivery days    : 1

Late deliveries: 715
Late percentage: 28.3%


**Block 6 — Monthly aggregation**

In [139]:
monthly_revenue = (
    orders_clean
    .groupby(orders_clean["order_date"].dt.to_period("M"))["amount"]
    .sum()
    .reset_index()
)

monthly_revenue.columns = ["month", "total_revenue"]
monthly_revenue["total_revenue"] = monthly_revenue["total_revenue"].round(2)
print(monthly_revenue.tail(10))

      month  total_revenue
26  2024-03     5479928.21
27  2024-04     5420537.23
28  2024-05     6784684.45
29  2024-06     5682686.41
30  2024-07     4582397.80
31  2024-08     5184068.11
32  2024-09     5108498.64
33  2024-10     5841810.73
34  2024-11     5311784.71
35  2024-12     5847564.90


**Practice tasks**

In [140]:
# Task 1
# From orders_clean:
# How many orders were placed in each quarter of 2024?
# Expected output — quarter number and order count
# Hint — filter 2024 first, then groupby quarter column

quarter_count = (
    orders_2024.groupby("quarter")["order_id"]
    .count()
    .reset_index(name="order_count")
)
print(quarter_count)

   quarter  order_count
0      1.0          277
1      2.0          333
2      3.0          277
3      4.0          283


In [141]:
# Task 2
# For delivered orders only:
# What is the average delivery days?
# How many were delivered within 5 days?
# What percentage is that?

delivered = orders_clean[
    orders_clean['status'] == "delivered"
].copy()

delivered["delivery_days"] = (
    delivered["delivery_date"] - delivered["order_date"]
).dt.days

print(f"Average delivery days: {delivered['delivery_days'].mean():.1f}")

# Flag late deliveries — more than 10 days
delivered["within_5_days"] = delivered["delivery_days"] < 6
print(f"\nFast deliveries: {delivered['within_5_days'].sum()}")
print(f"Fast Delivery percentage: {delivered['within_5_days'].mean()*100:.1f}%")

Average delivery days: 7.4

Fast deliveries: 911
Fast Delivery percentage: 36.0%


In [142]:
# Task 3
# Which month of the year has the highest
# total revenue across all years?
# Expected — month number and total revenue
# Hint — groupby month column, sum amount,
#         sort descending, show top 3

monthly_revenue = (
    orders_clean
    .groupby(orders_clean["order_date"].dt.month)["amount"]
    .sum()
    .reset_index()
)

monthly_revenue.columns = ["month", "total_revenue"]
monthly_revenue["total_revenue"] = monthly_revenue["total_revenue"].round(2)
monthly_revenue = monthly_revenue.sort_values(
    "total_revenue", ascending=False
)
print(monthly_revenue.head(3))

    month  total_revenue
4       5    19041291.75
5       6    17584305.63
11     12    17487539.22


**Section 3.7 — Transformations**

**What this section does**

Teaches you how to create new columns and modify existing ones based on calculations and conditions. Every pipeline transforms raw data into something more useful — adding a tier column, calculating discounts, flagging high value records.

**What it does to the data**

Adds new columns or modifies existing ones. The number of rows stays the same. Only columns change.

**What to expect from the output**

Same DataFrame with additional calculated columns — revenue after tax, customer tier, high value flag, discount amount.

**Block 1 — Adding a simple calculated column**


In [143]:
# Add tax amount and final price columns
orders_clean["tax_amount"] = orders_clean["amount"] * 0.18
orders_clean["final_amount"] = orders_clean["amount"] + orders_clean["tax_amount"]

print(orders_clean[["order_id", "amount", "tax_amount", "final_amount"]].head(5))

  order_id    amount  tax_amount  final_amount
1    10001  13068.13   2352.2634    15420.3934
2    10002  29699.84   5345.9712    35045.8112
3    10003  29958.61   5392.5498    35351.1598
4    10004  32484.07   5847.1326    38331.2026
5    10005  26636.20   4794.5160    31430.7160


**Block 2 — assign() — adding columns cleanly in a chain**


In [144]:
orders_clean = orders_clean.assign(
    discount_amount = orders_clean["amount"] * 0.10,
    discounted_price = orders_clean["amount"] * 0.90
)

print(orders_clean[["order_id", "amount",
                     "discount_amount",
                     "discounted_price"]].head(5))

  order_id    amount  discount_amount  discounted_price
1    10001  13068.13         1306.813         11761.317
2    10002  29699.84         2969.984         26729.856
3    10003  29958.61         2995.861         26962.749
4    10004  32484.07         3248.407         29235.663
5    10005  26636.20         2663.620         23972.580


**Block 3 — np.where() — conditional column**

This is the most important transformation pattern in DE. Creates a column based on a condition — like a one-line if/else applied to every row:

In [145]:
import numpy as np

# Flag high value orders
orders_clean["is_high_value"] = np.where(
    orders_clean["amount"] > 40000,
    "high",
    "normal"
)

print(orders_clean["is_high_value"].value_counts())

is_high_value
normal    2588
high      2378
Name: count, dtype: int64


**Block 4 — np.select() — multiple conditions**

When you have more than two outcomes use np.select():

In [146]:
# Customer tier based on total_spent
conditions = [
    customers_clean["total_spent"] > 150000,
    customers_clean["total_spent"] > 75000,
    customers_clean["total_spent"] > 25000,
    customers_clean["total_spent"] > 5000
]

choices = ["Platinum", "Gold", "Silver", "Bronze"]

customers_clean["tier"] = np.select(
    conditions,
    choices,
    default="Basic"
)

print(customers_clean["tier"].value_counts())


tier
Gold        379
Silver      249
Platinum    233
Bronze       99
Basic        26
Name: count, dtype: int64


**Block 5 — map() — value substitution**

Replace values using a dictionary lookup:

In [147]:
# Replace status codes with readable labels
status_labels = {
    "pending"   : "Order Placed",
    "shipped"   : "In Transit",
    "delivered" : "Completed",
    "cancelled" : "Cancelled",
    "returned"  : "Returned"
}

orders_clean["status_label"] = orders_clean["status"].map(status_labels)

print(orders_clean[["order_id", "status", "status_label"]].head(8))

  order_id     status  status_label
1    10001   returned      Returned
2    10002    shipped    In Transit
3    10003   returned      Returned
4    10004    shipped    In Transit
5    10005    shipped    In Transit
6    10006  delivered     Completed
7    10007  delivered     Completed
8    10008    pending  Order Placed


**Block 6 — apply() — when nothing else works**

apply() runs a function on each row or column. It is slower than vectorised operations so use it only when np.where and np.select cannot handle the logic:

In [148]:
def classify_delivery(row):
    if pd.isna(row["delivery_days"]):
        return "no delivery"
    elif row["delivery_days"] <= 3:
        return "express"
    elif row["delivery_days"] <= 7:
        return "standard"
    else:
        return "delayed"

delivered["delivery_class"] = delivered.apply(
    classify_delivery,
    axis=1
)

print(delivered["delivery_class"].value_counts())

delivery_class
delayed     1249
standard     719
express      562
Name: count, dtype: int64


**Practice tasks**


In [149]:
# Task 1
# On orders_clean add these three columns:
# - amount_with_gst  — amount plus 18% GST
# - is_premium_order — True if amount > 50000, False otherwise
#                      use np.where
# - order_size — "large" if amount > 30000
#                "medium" if amount > 10000
#                "small" for everything else
#                use np.select
# Print head(5) showing order_id, amount,
# amount_with_gst, is_premium_order, order_size
import numpy as np

# amount with GST
orders_clean["amount_with_gst"] = orders_clean["amount"] * 1.18

# is premium order
orders_clean["is_premium_order"] = np.where(
    orders_clean["amount"] > 50000,
    True,
    False
)

# order size
conditions = [
    orders_clean["amount"] > 30000,
    orders_clean["amount"] > 10000,
]
choices = ["large", "medium"]

orders_clean["order_size"] = np.select(
    conditions,
    choices,
    default="small"
)

print(orders_clean[[
    "order_id", "amount",
    "amount_with_gst",
    "is_premium_order",
    "order_size"
]].head(5))

  order_id    amount  amount_with_gst  is_premium_order order_size
1    10001  13068.13       15420.3934             False     medium
2    10002  29699.84       35045.8112             False     medium
3    10003  29958.61       35351.1598             False     medium
4    10004  32484.07       38331.2026             False      large
5    10005  26636.20       31430.7160             False     medium


In [150]:
# Task 2
# On customers_clean:
# - Add a tier column using np.select
#   Platinum — total_spent > 150000
#   Gold     — total_spent > 75000
#   Silver   — total_spent > 25000
#   Bronze   — total_spent > 5000
#   Basic    — everything else
# - Add a country_code column using map()
#   India     → IN
#   UAE       → AE
#   USA       → US
#   UK        → GB
#   Singapore → SG
# Print tier value_counts and
# country_code value_counts

conditions = [
    customers_clean["total_spent"] > 150000,
    customers_clean["total_spent"] > 75000,
    customers_clean["total_spent"] > 25000,
    customers_clean["total_spent"] > 5000
]

choices = ["Platinum", "Gold", "Silver", "Bronze"]

customers_clean["tier"] = np.select(
    conditions,
    choices,
    default="Basic"
)

print(customers_clean["tier"].value_counts())

country_labels = {
    "India"   : "IN",
    "UAE"   : "AE",
    "USA" : "US",
    "UK" : "GB",
    "Singapore"  : "SG"
}

customers_clean["country_code"] = customers_clean["country"].map(country_labels)
print(customers_clean["country_code"].value_counts())
print(customers_clean[["country", "country_code"]].head(8))

tier
Gold        379
Silver      249
Platinum    233
Bronze       99
Basic        26
Name: count, dtype: int64
country_code
US    210
GB    205
SG    200
AE    191
IN    180
Name: count, dtype: int64
     country country_code
1      India           IN
2        UAE           AE
3      India           IN
4        USA           US
5  Singapore           SG
6         UK           GB
7        USA           US
8        USA           US


In [151]:
# Task 3
# On delivered DataFrame:
# You already have delivery_days column
# Add delivery_class using apply()
#   express  — delivery_days <= 3
#   standard — delivery_days <= 7
#   delayed  — everything else
# Print delivery_class value_counts
def classify_delivery(row):
    if pd.isna(row["delivery_days"]):
        return "no delivery"
    elif row["delivery_days"] <= 3:
        return "express"
    elif row["delivery_days"] <= 7:
        return "standard"
    else:
        return "delayed"

delivered["delivery_class"] = delivered.apply(
    classify_delivery,
    axis=1
)

print(delivered["delivery_class"].value_counts())

delivery_class
delayed     1249
standard     719
express      562
Name: count, dtype: int64


**Section 3.8 — GroupBy and Aggregations**

**What this section does**

Teaches you how to summarise data by groups. Revenue by country. Orders by status. Average spend by customer tier. This is the most used operation in analytics and DE work.

**What it does to the data**

Collapses many rows into summary rows. 5000 order rows become 5 rows — one per status. The detail is gone, the summary remains.

**What to expect from the output**

A smaller DataFrame with one row per group and calculated values — sums, counts, averages, min, max.

In [152]:
# Total revenue by status
revenue_by_status = (
    orders_clean
    .groupby("status")["amount"]
    .sum()
    .reset_index()
)

revenue_by_status.columns = ["status", "total_revenue"]
revenue_by_status = revenue_by_status.sort_values(
    "total_revenue", ascending=False
)
print(revenue_by_status)

      status  total_revenue
1  delivered   1.007814e+08
4    shipped   3.780360e+07
0  cancelled   2.940512e+07
2    pending   1.961835e+07
3   returned   9.803446e+06


**Block 2 — Multiple aggregations at once**

In [153]:
# Multiple stats in one operation
order_summary = (
    orders_clean
    .groupby("status")["amount"]
    .agg(
        order_count  = "count",
        total_revenue= "sum",
        avg_amount   = "mean",
        min_amount   = "min",
        max_amount   = "max"
    )
    .reset_index()
)

order_summary = order_summary.round(2)
print(order_summary)

      status  order_count  total_revenue  avg_amount  min_amount  max_amount
0  cancelled          735   2.940512e+07    40006.97      287.52    79742.12
1  delivered         2530   1.007814e+08    39834.53      243.33    79942.60
2    pending          496   1.961835e+07    39553.13      420.49    79873.90
3   returned          245   9.803446e+06    40014.07      322.40    78225.73
4    shipped          960   3.780360e+07    39378.75      243.47    79913.01


**Block 3 — Groupby on two columns**


In [154]:
# Revenue by warehouse and status
warehouse_status = (
    orders_clean
    .groupby(["warehouse", "status"])["amount"]
    .sum()
    .reset_index()
)

warehouse_status.columns = ["warehouse", "status", "total_revenue"]
warehouse_status = warehouse_status.sort_values(
    "total_revenue", ascending=False
)
print(warehouse_status.head(10))

    warehouse     status  total_revenue
1       Delhi  delivered    21989732.24
11     London  delivered    20320527.31
21  Singapore  delivered    19799637.73
6       Dubai  delivered    19494130.05
16     Mumbai  delivered    19177334.19
14     London    shipped     8144138.00
24  Singapore    shipped     7997295.69
19     Mumbai    shipped     7939571.03
9       Dubai    shipped     7078383.12
10     London  cancelled     6765306.00


**Block 4 — transform() — the most misunderstood operation**

agg() collapses rows into groups. transform() keeps the same number of rows but fills each row with a group-level value.

In [155]:
# Add total revenue per warehouse to each order row
orders_clean["warehouse_total"] = (
    orders_clean
    .groupby("warehouse")["amount"]
    .transform("sum")
)

# Now each order knows its warehouse's total revenue
print(orders_clean[[
    "order_id", "warehouse",
    "amount", "warehouse_total"
]].head(8))

orders_clean["pct_of_warehouse"] = (
    orders_clean["amount"] / orders_clean["warehouse_total"] * 100
).round(2)

print(orders_clean[[
    "order_id", "warehouse",
    "amount", "warehouse_total", "pct_of_warehouse"
]].head(8))

  order_id  warehouse    amount  warehouse_total
1    10001  Singapore  13068.13      38842655.27
2    10002      Dubai  29699.84      38533130.05
3    10003      Dubai  29958.61      38533130.05
4    10004      Dubai  32484.07      38533130.05
5    10005  Singapore  26636.20      38842655.27
6    10006      Delhi  27227.02      40688986.42
7    10007      Dubai  24784.96      38533130.05
8    10008      Delhi  48730.47      40688986.42
  order_id  warehouse    amount  warehouse_total  pct_of_warehouse
1    10001  Singapore  13068.13      38842655.27              0.03
2    10002      Dubai  29699.84      38533130.05              0.08
3    10003      Dubai  29958.61      38533130.05              0.08
4    10004      Dubai  32484.07      38533130.05              0.08
5    10005  Singapore  26636.20      38842655.27              0.07
6    10006      Delhi  27227.02      40688986.42              0.07
7    10007      Dubai  24784.96      38533130.05              0.06
8    10008      Delhi  

**Block 5 — filter() on groups**

Keep only groups that meet a condition:

In [156]:
# Keep only warehouses with total revenue above 10 million
high_revenue_warehouses = (
    orders_clean
    .groupby("warehouse")
    .filter(lambda x: x["amount"].sum() > 10000000)
)

print(f"Original rows : {len(orders_clean)}")
print(f"Filtered rows : {len(high_revenue_warehouses)}")
print(high_revenue_warehouses["warehouse"].unique())

Original rows : 4966
Filtered rows : 4966
['Singapore' 'Dubai' 'Delhi' 'London' 'Mumbai']


In [157]:
# Task 1
# From orders_clean:
# For each payment_method calculate:
# - number of orders
# - total revenue
# - average order amount
# - maximum order amount
# Sort by total revenue descending
# Round all numbers to 2 decimal places

#revenue_by_payment_method = (
#   orders_clean
#    .groupby("payment_method")["amount"]
#   .sum()
#    .reset_index()
#)

payment_method_summary = (
    orders_clean
    .groupby("payment_method")["amount"]
    .agg(
        order_count  = "count",
        total_revenue= "sum",
        avg_amount   = "mean",
        min_amount   = "min",
        max_amount   = "max"
    )
    .reset_index()
)

payment_method_summary = payment_method_summary.round(2)
print(payment_method_summary)

  payment_method  order_count  total_revenue  avg_amount  min_amount  \
0            UPI         1023    39956333.52    39058.00      243.33   
1    credit_card          994    39084190.29    39320.11      273.53   
2     debit_card         1003    41294892.25    41171.38      243.47   
3    net_banking          976    38753293.52    39706.24      261.38   
4         wallet          970    38323172.50    39508.43      265.47   

   max_amount  
0    79875.26  
1    79942.60  
2    79849.30  
3    79913.01  
4    79872.73  


In [158]:
# Task 2
# From orders_clean:
# Add a column called "status_avg_amount"
# containing the average amount for that order's status
# So every "delivered" order gets the average
# amount of all delivered orders
# Use transform()
# Print head(5) showing order_id, status,
# amount, status_avg_amount

orders_clean["status_avg_amount"] = (
    orders_clean
    .groupby("status")["amount"]
    .transform("mean")
)

# Now each order knows its warehouse's total revenue
print(orders_clean[[
    "order_id", "status",
    "amount", "status_avg_amount"
]].head(8))

  order_id     status    amount  status_avg_amount
1    10001   returned  13068.13       40014.067306
2    10002    shipped  29699.84       39378.750844
3    10003   returned  29958.61       40014.067306
4    10004    shipped  32484.07       39378.750844
5    10005    shipped  26636.20       39378.750844
6    10006  delivered  27227.02       39834.530245
7    10007  delivered  24784.96       39834.530245
8    10008    pending  48730.47       39553.125887


In [159]:
# Task 3
# From customers_clean:
# Group by country and tier
# Calculate count of customers and
# average total_spent per group
# Sort by average total_spent descending
# Print top 10 rows

country_tier = (
    customers_clean
    .groupby(['country', 'tier'])['total_spent']
    .agg(
        customer_count = "count",
        avg_spent = "mean"
    )
    .reset_index()
)
country_tier["avg_spent"] = country_tier["avg_spent"].round(2)
country_tier = country_tier.sort_values("avg_spent", ascending=False)
print(country_tier.head(10))

      country      tier  customer_count  avg_spent
23        USA  Platinum              55  176963.78
13        UAE  Platinum              48  176741.56
3       India  Platinum              43  176064.19
8   Singapore  Platinum              41  173892.12
18         UK  Platinum              46  173528.87
17         UK      Gold              87  115777.22
2       India      Gold              76  113653.36
22        USA      Gold              84  111868.61
12        UAE      Gold              56  111800.91
7   Singapore      Gold              76  108339.16


/tmp/ipykernel_6994/3417537484.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['country', 'tier'])['total_spent']


**Section 3.9 — Reshaping Data**

**What this section does**

Teaches you how to change the shape of a DataFrame — turning wide data into long data and vice versa. This is needed constantly when preparing data for BI tools, dashboards, and analytical models.

**What it does to the data**

Does not add or remove information. Reorganises the same data into a different structure. Wide tables become tall. Tall tables become wide.

**What to expect from the output**

A DataFrame with different dimensions — more rows and fewer columns, or fewer rows and more columns — containing the same underlying data.

**Block 1 — Create sample wide data**


In [160]:
import pandas as pd
import numpy as np

# Wide format — quarterly revenue per warehouse
wide_data = pd.DataFrame({
    "warehouse" : ["Mumbai", "Delhi", "Dubai", "London"],
    "Q1_revenue": [2500000, 1800000, 3200000, 2100000],
    "Q2_revenue": [2800000, 2100000, 3500000, 2400000],
    "Q3_revenue": [2600000, 1900000, 3100000, 2200000],
    "Q4_revenue": [3100000, 2300000, 3800000, 2700000]
})

print("Wide format:")
print(wide_data)
print(f"Shape: {wide_data.shape}")

Wide format:
  warehouse  Q1_revenue  Q2_revenue  Q3_revenue  Q4_revenue
0    Mumbai     2500000     2800000     2600000     3100000
1     Delhi     1800000     2100000     1900000     2300000
2     Dubai     3200000     3500000     3100000     3800000
3    London     2100000     2400000     2200000     2700000
Shape: (4, 5)


**Block 2 — melt() — wide to long**

In [161]:
# Convert wide to long
long_data = wide_data.melt(
    id_vars    = ["warehouse"],
    value_vars = ["Q1_revenue", "Q2_revenue",
                  "Q3_revenue", "Q4_revenue"],
    var_name   = "quarter",
    value_name = "revenue"
)

print("\nLong format:")
print(long_data.sort_values("warehouse"))
print(f"Shape: {long_data.shape}")


Long format:
   warehouse     quarter  revenue
1      Delhi  Q1_revenue  1800000
5      Delhi  Q2_revenue  2100000
9      Delhi  Q3_revenue  1900000
13     Delhi  Q4_revenue  2300000
2      Dubai  Q1_revenue  3200000
6      Dubai  Q2_revenue  3500000
10     Dubai  Q3_revenue  3100000
14     Dubai  Q4_revenue  3800000
3     London  Q1_revenue  2100000
7     London  Q2_revenue  2400000
11    London  Q3_revenue  2200000
15    London  Q4_revenue  2700000
0     Mumbai  Q1_revenue  2500000
4     Mumbai  Q2_revenue  2800000
8     Mumbai  Q3_revenue  2600000
12    Mumbai  Q4_revenue  3100000
Shape: (16, 3)


**Block 3 — Clean up the quarter column**

In [162]:
# Remove "_revenue" suffix from quarter names
long_data["quarter"] = long_data["quarter"].str.replace(
    "_revenue", "", regex=False
)

print(long_data.sort_values(["warehouse", "quarter"]))

   warehouse quarter  revenue
1      Delhi      Q1  1800000
5      Delhi      Q2  2100000
9      Delhi      Q3  1900000
13     Delhi      Q4  2300000
2      Dubai      Q1  3200000
6      Dubai      Q2  3500000
10     Dubai      Q3  3100000
14     Dubai      Q4  3800000
3     London      Q1  2100000
7     London      Q2  2400000
11    London      Q3  2200000
15    London      Q4  2700000
0     Mumbai      Q1  2500000
4     Mumbai      Q2  2800000
8     Mumbai      Q3  2600000
12    Mumbai      Q4  3100000


**Block 4 — pivot_table() — long to wide**

The reverse operation — collapse long data into a wide summary:

In [163]:
# Revenue by warehouse and quarter — wide format
pivot = long_data.pivot_table(
    index   = "warehouse",
    columns = "quarter",
    values  = "revenue",
    aggfunc = "sum"
)

print("\nPivot table:")
print(pivot)


Pivot table:
quarter         Q1       Q2       Q3       Q4
warehouse                                    
Delhi      1800000  2100000  1900000  2300000
Dubai      3200000  3500000  3100000  3800000
London     2100000  2400000  2200000  2700000
Mumbai     2500000  2800000  2600000  3100000


**Block 5 — pivot_table() on real TradeSphere data**

In [164]:
# Monthly revenue by warehouse — real data
monthly_warehouse = orders_clean.pivot_table(
    index   = orders_clean["order_date"].dt.month,
    columns = "warehouse",
    values  = "amount",
    aggfunc = "sum"
).round(2)

monthly_warehouse.index.name = "month"
print(monthly_warehouse)

warehouse       Delhi       Dubai      London      Mumbai   Singapore
month                                                                
1          3397198.54  2604799.43  3339513.81  3032823.49  3595841.50
2          3232858.51  2130731.09  3611584.94  3501027.13  2526993.27
3          3696845.79  2568275.91  3792420.69  2962311.72  3889515.52
4          3887227.82  3340863.73  3333462.13  3304227.49  2674033.19
5          3737773.52  4059167.37  3864126.89  3600389.32  3779834.65
6          2942941.87  3550733.39  3920360.66  3104684.69  4065585.02
7          3870579.17  3284665.67  2967593.40  2645389.46  3176660.68
8          3308372.21  3236367.79  3344887.59  2860952.71  2401012.61
9          3050854.14  3446200.12  3208087.28  2963381.01  3578641.43
10         3139842.78  3644178.66  3545683.25  3018771.77  2876431.14
11         3455270.54  2924642.46  3001808.16  3292558.84  2633356.91
12         2969221.53  3742504.43  3211959.99  3919103.92  3644749.35


**Block 6 — stack() and unstack()**


In [165]:
# stack() — moves column headers into rows
stacked = monthly_warehouse.stack().reset_index()
stacked.columns = ["month", "warehouse", "revenue"]
print("\nStacked:")
print(stacked.head(8))

# unstack() — moves row index into columns
# This is the reverse — back to wide
unstacked = stacked.set_index(
    ["month", "warehouse"]
)["revenue"].unstack("warehouse")
print("\nUnstacked:")
print(unstacked.head(5))


Stacked:
   month  warehouse     revenue
0      1      Delhi  3397198.54
1      1      Dubai  2604799.43
2      1     London  3339513.81
3      1     Mumbai  3032823.49
4      1  Singapore  3595841.50
5      2      Delhi  3232858.51
6      2      Dubai  2130731.09
7      2     London  3611584.94

Unstacked:
warehouse       Delhi       Dubai      London      Mumbai   Singapore
month                                                                
1          3397198.54  2604799.43  3339513.81  3032823.49  3595841.50
2          3232858.51  2130731.09  3611584.94  3501027.13  2526993.27
3          3696845.79  2568275.91  3792420.69  2962311.72  3889515.52
4          3887227.82  3340863.73  3333462.13  3304227.49  2674033.19
5          3737773.52  4059167.37  3864126.89  3600389.32  3779834.65


**Practice tasks**


In [166]:
# Task 1
# Create this wide DataFrame:
wide_orders = pd.DataFrame({
    "customer_id"       : ["C001", "C002", "C003"],
    "electronics_spend" : [45000, 12000, 78000],
    "clothing_spend"    : [8000,  15000, 6000],
    "books_spend"       : [2000,  5000,  3000],
    "food_spend"        : [12000, 8000,  20000]
})

# Use melt() to convert to long format
# id_vars    — customer_id
# value_vars — the four spend columns
# var_name   — "category"
# value_name — "amount"
# Print the result
# How many rows does the long format have?

long_data = wide_orders.melt(
    id_vars    = ["customer_id"],
    value_vars = ["electronics_spend", "clothing_spend",
                  "books_spend", "food_spend"],
    var_name   = "category",
    value_name = "amount"
)

print("\nLong format:")
print(long_data.sort_values("category"))
print(f"Shape: {long_data.shape}")


Long format:
   customer_id           category  amount
6         C001        books_spend    2000
7         C002        books_spend    5000
8         C003        books_spend    3000
3         C001     clothing_spend    8000
4         C002     clothing_spend   15000
5         C003     clothing_spend    6000
0         C001  electronics_spend   45000
1         C002  electronics_spend   12000
2         C003  electronics_spend   78000
9         C001         food_spend   12000
10        C002         food_spend    8000
11        C003         food_spend   20000
Shape: (12, 3)


In [167]:
# Task 2
# Using orders_clean:
# Create a pivot table showing
# total revenue for each combination of
# payment_method (rows) and
# warehouse (columns)
# Round to 2 decimal places
# Print the result

pivot = orders_clean.pivot_table(
    index   = "payment_method",
    columns = "warehouse",
    values  = "amount",
    aggfunc = "sum"
)

print("\nPivot table:")
print(pivot)


Pivot table:
warehouse            Delhi       Dubai      London      Mumbai   Singapore
payment_method                                                            
UPI             7637252.74  7691345.28  8082675.94  8210568.94  8334490.62
credit_card     8616609.47  7794122.58  8852964.28  6880209.36  6940284.60
debit_card      8793734.21  8183489.58  8327177.74  7671248.68  8319242.04
net_banking     8074500.45  7076236.39  8125439.92  8239726.88  7237389.88
wallet          7566889.55  7787936.22  7753230.91  7203867.69  8011248.13


In [168]:
# Task 3
# Using orders_clean:
# Create a pivot table showing
# order count for each
# status (rows) and
# order_size column you created in 3.7 (columns)
# Use aggfunc="count" and values="order_id"
# Print the result

pivot = orders_clean.pivot_table(
    index   = "status",
    columns = "order_size",
    values  = "order_id",
    aggfunc = "count"
)

print("\nPivot table:")
print(pivot)


Pivot table:
order_size  large  medium  small
status                          
cancelled     457     176    102
delivered    1579     638    313
pending       312     126     58
returned      158      63     24
shipped       584     262    114


**Section 3.10 — Merging Data**

**What this section does**

Teaches you how to combine two or more DataFrames together. In real pipelines data never lives in one place — customers are in one table, orders in another, products in a third. Merging brings them together for analysis.

**What it does to the data**

Combines rows from two DataFrames based on a matching column. The result has columns from both DataFrames side by side.

**What to expect from the output**

A wider DataFrame — more columns — containing data from both source tables joined on a common key.

**The four join types — understand these first**

**INNER JOIN  — only rows that match in BOTH tables**

**LEFT JOIN   — all rows from left, matching from right**

**RIGHT JOIN  — all rows from right, matching from left**

**OUTER JOIN  — all rows from both tables**

**Full worked example**

**Block 1 — Basic inner merge**


In [169]:
# Merge orders with customers
# Match on customer_id

orders_with_customers = orders_clean.merge(
    customers_clean[["customer_id", "name",
                     "country", "tier"]],
    on = "customer_id",
    how = "inner"
)

print(f"Orders rows      : {len(orders_clean)}")
print(f"After merge rows : {len(orders_with_customers)}")
print(orders_with_customers[[
    "order_id", "customer_id",
    "name", "country", "amount"
]].head(5))


Orders rows      : 4966
After merge rows : 4886
  order_id customer_id            name    country    amount
0    10001        1119      Kavya Iyer         UK  13068.13
1    10002        1575     Rahul Kumar      India  29699.84
2    10003        1552      Kavya Nair  Singapore  29958.61
3    10004        1786  Vikram Johnson        UAE  32484.07
4    10005        1302       Emma Shah         UK  26636.20


**Block 2 — Left merge**


In [170]:
# Left merge — keep all orders even if customer not found
orders_left = orders_clean.merge(
    customers_clean[["customer_id", "name", "country"]],
    on  = "customer_id",
    how = "left"
)

print(f"Orders rows  : {len(orders_clean)}")
print(f"After merge  : {len(orders_left)}")

# Check for orders with no matching customer
no_customer = orders_left[orders_left["name"].isna()]
print(f"Orders with no customer: {len(no_customer)}")

Orders rows  : 4966
After merge  : 4966
Orders with no customer: 80


**Block 3 — Merging on different column names**


In [171]:
# If columns had different names you would use left_on and right_on
# Example — if orders had "cust_id" and customers had "customer_id"
orders_renamed = orders_clean.rename(
    columns={"customer_id": "cust_id"}
)

merged = orders_renamed.merge(
    customers_clean[["customer_id", "name"]],
    left_on  = "cust_id",
    right_on = "customer_id",
    how      = "left"
)

print(merged[["order_id", "cust_id",
              "customer_id", "name"]].head(3))

  order_id cust_id customer_id         name
0    10001    1119        1119   Kavya Iyer
1    10002    1575        1575  Rahul Kumar
2    10003    1552        1552   Kavya Nair


**Block 4 — Three table merge**


In [172]:
# Orders + Customers + Products in one chain
full_orders = (
    orders_clean
    .merge(
        customers_clean[["customer_id", "name",
                         "country", "tier"]],
        on  = "customer_id",
        how = "left"
    )
    .merge(
        products[["product_id", "name",
                  "category", "price"]],
        on       = "product_id",
        how      = "left",
        suffixes = ("_customer", "_product")
    )
)

print(f"Final shape: {full_orders.shape}")
print(full_orders[[
    "order_id", "name_customer",
    "name_product", "category", "amount"
]].head(5))

Final shape: (4966, 31)
  order_id   name_customer name_product category    amount
0    10001      Kavya Iyer   Product_10     food  13068.13
1    10002     Rahul Kumar  Product_181     home  29699.84
2    10003      Kavya Nair   Product_41     home  29958.61
3    10004  Vikram Johnson   Product_37    books  32484.07
4    10005       Emma Shah  Product_121     home  26636.20


**Block 5 — concat() — stacking DataFrames**


In [173]:
# Split orders by year then recombine
orders_2022 = orders_clean[
    orders_clean["order_date"].dt.year == 2022
]
orders_2023 = orders_clean[
    orders_clean["order_date"].dt.year == 2023
]
orders_2024 = orders_clean[
    orders_clean["order_date"].dt.year == 2024
]

print(f"2022: {len(orders_2022)}")
print(f"2023: {len(orders_2023)}")
print(f"2024: {len(orders_2024)}")

# Stack them back together
all_orders = pd.concat(
    [orders_2022, orders_2023, orders_2024],
    ignore_index = True
)

print(f"Combined: {len(all_orders)}")

2022: 1691
2023: 1614
2024: 1661
Combined: 4966


**Block 6 — Check for merge duplicates**


In [174]:
# Always check for duplicate rows after merging
print(f"Before merge: {len(orders_clean)}")
print(f"After merge : {len(orders_with_customers)}")

# If after > before you have duplicates in the right table
# Check with indicator=True
orders_check = orders_clean.merge(
    customers_clean[["customer_id", "name"]],
    on        = "customer_id",
    how       = "left",
    indicator = True
)

print(orders_check["_merge"].value_counts())

Before merge: 4966
After merge : 4886
_merge
both          4886
left_only       80
right_only       0
Name: count, dtype: int64


In [181]:
# Task 1
# Merge orders_clean with customers_clean
# Keep these columns from customers:
# customer_id, name, country, tier
# Use left merge on customer_id
# After merging:
# - Print shape
# - Print how many orders have no matching customer
# - Print top 5 rows showing
#   order_id, name, country, amount, status

orders_left = orders_clean.merge(
    customers_clean[["customer_id", "name", "country", "tier"]],
    on  = "customer_id",
    how = "left"
)

print(f"Orders rows  : {len(orders_clean)}")
print(f"After merge  : {len(orders_left)}")

# Check for orders with no matching customer
no_customer = orders_left[orders_left["name"].isna()]
print(f"Orders with no customer: {len(no_customer)}")

print(orders_left[["order_id", "name", "country", "amount", "status"]].head(5))


Orders rows  : 4966
After merge  : 4966
Orders with no customer: 80
  order_id            name    country    amount    status
0    10001      Kavya Iyer         UK  13068.13  returned
1    10002     Rahul Kumar      India  29699.84   shipped
2    10003      Kavya Nair  Singapore  29958.61  returned
3    10004  Vikram Johnson        UAE  32484.07   shipped
4    10005       Emma Shah         UK  26636.20   shipped


In [184]:
# Task 2
# Using the merged DataFrame from Task 1:
# Add product information — merge with products
# Keep from products: product_id, name, category, price
# Use suffixes ("_customer", "_product") for name column
# After merging:
# - Print shape
# - Show top 5 rows with
#   order_id, name_customer, name_product,
#   category, amount

orders_left_full = orders_left. merge(

        products[["product_id", "name",
                  "category", "price"]],
        on       = "product_id",
        how      = "left",
        suffixes = ("_customer", "_product")
    )

print(f"Final shape: {full_orders.shape}")
print(orders_left_full[["order_id", "name_customer",
    "name_product", "category", "amount"]].head(5))

Final shape: (4966, 31)
  order_id   name_customer name_product category    amount
0    10001      Kavya Iyer   Product_10     food  13068.13
1    10002     Rahul Kumar  Product_181     home  29699.84
2    10003      Kavya Nair   Product_41     home  29958.61
3    10004  Vikram Johnson   Product_37    books  32484.07
4    10005       Emma Shah  Product_121     home  26636.20


In [185]:
# Task 3
# Using the full merged DataFrame from Task 2:
# Group by country and category
# Calculate total revenue per combination
# Sort descending
# Print top 10 rows
# Which country + category drives the most revenue?
country_category = (
    orders_left_full
    .groupby(["country", "category"])["amount"]
    .sum()
    .reset_index()
)

country_category.columns = ["country", "category", "total_revenue"]
country_category["total_revenue"] = country_category["total_revenue"].round(2)
country_category = country_category.sort_values(
    "total_revenue", ascending=False
)

print(country_category.head(10))

      country     category  total_revenue
17         UK  electronics    10849312.42
7   Singapore  electronics     9916840.85
8   Singapore         food     9914141.03
22        USA  electronics     9736121.10
2       India  electronics     9426078.66
12        UAE  electronics     9258876.10
23        USA         food     8859992.94
18         UK         food     8751262.32
13        UAE         food     8696083.36
24        USA         home     8582369.15


/tmp/ipykernel_6994/2692872132.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["country", "category"])["amount"]


**Section 3.11 — Window Operations**

**What this section does**

Teaches you how to calculate values that look across multiple rows — moving averages, running totals, rankings, and comparing each row to the previous one. These are the Pandas equivalents of SQL window functions.

**What it does to the data**

Adds new columns calculated from a sliding window of rows. The number of rows stays the same — like transform(). But instead of group-level values, you get calculations that move across ordered rows.

**What to expect from the output**

New columns containing — cumulative totals that grow with each row, rolling averages that smooth out fluctuations, rank numbers within groups, and lag values showing what the previous row contained.

**Block 1 — rolling() — moving average**


In [186]:
# Monthly revenue first
monthly = (
    orders_clean
    .groupby(orders_clean["order_date"].dt.to_period("M"))["amount"]
    .sum()
    .reset_index()
)
monthly.columns = ["month", "revenue"]
monthly = monthly.sort_values("month")

# 3-month rolling average
monthly["rolling_3m"] = monthly["revenue"].rolling(
    window = 3
).mean().round(2)

print(monthly.tail(10))

      month     revenue  rolling_3m
26  2024-03  5479928.21  5130895.57
27  2024-04  5420537.23  5314724.80
28  2024-05  6784684.45  5895049.96
29  2024-06  5682686.41  5962636.03
30  2024-07  4582397.80  5683256.22
31  2024-08  5184068.11  5149717.44
32  2024-09  5108498.64  4958321.52
33  2024-10  5841810.73  5378125.83
34  2024-11  5311784.71  5420698.03
35  2024-12  5847564.90  5667053.45


**Block 2 — cumsum() — running total**

In [187]:
monthly["cumulative_revenue"] = monthly["revenue"].cumsum().round(2)

print(monthly[["month", "revenue", "cumulative_revenue"]].tail(10))

      month     revenue  cumulative_revenue
26  2024-03  5479928.21        1.476478e+08
27  2024-04  5420537.23        1.530684e+08
28  2024-05  6784684.45        1.598531e+08
29  2024-06  5682686.41        1.655358e+08
30  2024-07  4582397.80        1.701182e+08
31  2024-08  5184068.11        1.753022e+08
32  2024-09  5108498.64        1.804107e+08
33  2024-10  5841810.73        1.862525e+08
34  2024-11  5311784.71        1.915643e+08
35  2024-12  5847564.90        1.974119e+08


**Block 3 — rank() — ranking within groups**


In [189]:
# Rank customers by total_spent within each country
customers_clean["rank_in_country"] = (
    customers_clean
    .groupby("country")["total_spent"]
    .rank(method="dense", ascending=False)
)

# Show top 3 per country
top_per_country = customers_clean[
    customers_clean["rank_in_country"] <= 3
][["customer_id", "name", "country",
   "total_spent", "rank_in_country"]].sort_values(["country", "rank_in_country"])

print(top_per_country)

    customer_id           name    country  total_spent  rank_in_country
40         1040     Arjun Chen      India    199856.85              1.0
961        1961    Vikram Wang      India    197654.56              2.0
549        1549    Meera Jones      India    196631.59              3.0
395        1395    Divya Jones  Singapore    198699.95              1.0
417        1417   Karan Garcia  Singapore    198069.55              2.0
43         1043     Rajesh Kim  Singapore    191484.87              3.0
927        1927      Rohan Lee        UAE    199417.21              1.0
388        1388    Rahul Singh        UAE    198572.93              2.0
489        1489     Rohan Iyer        UAE    197960.26              3.0
89         1089      Ali Jones         UK    197584.54              1.0
653        1653  Pooja Johnson         UK    197444.72              2.0
397        1397      Riya Nair         UK    197146.26              3.0
639        1639    Priya Jones        USA    199886.16          

/tmp/ipykernel_6994/2060152938.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("country")["total_spent"]


**Block 4 — shift() — lag values**


In [190]:
# Month over month revenue change
monthly["prev_month_revenue"] = monthly["revenue"].shift(1)

monthly["mom_change"] = (
    monthly["revenue"] - monthly["prev_month_revenue"]
).round(2)

monthly["mom_pct_change"] = (
    monthly["revenue"] / monthly["prev_month_revenue"] - 1
).round(4) * 100

print(monthly[[
    "month", "revenue",
    "prev_month_revenue",
    "mom_change",
    "mom_pct_change"
]].tail(10))

      month     revenue  prev_month_revenue  mom_change  mom_pct_change
26  2024-03  5479928.21          5043708.97   436219.24            8.65
27  2024-04  5420537.23          5479928.21   -59390.98           -1.08
28  2024-05  6784684.45          5420537.23  1364147.22           25.17
29  2024-06  5682686.41          6784684.45 -1101998.04          -16.24
30  2024-07  4582397.80          5682686.41 -1100288.61          -19.36
31  2024-08  5184068.11          4582397.80   601670.31           13.13
32  2024-09  5108498.64          5184068.11   -75569.47           -1.46
33  2024-10  5841810.73          5108498.64   733312.09           14.35
34  2024-11  5311784.71          5841810.73  -530026.02           -9.07
35  2024-12  5847564.90          5311784.71   535780.19           10.09


**Block 5 — Combining window operations**


In [191]:
# Complete monthly analysis in one chain
monthly_analysis = (
    orders_clean
    .groupby(orders_clean["order_date"].dt.to_period("M"))["amount"]
    .sum()
    .reset_index()
)

monthly_analysis.columns = ["month", "revenue"]
monthly_analysis = monthly_analysis.sort_values("month")

monthly_analysis["rolling_3m_avg"]   = monthly_analysis["revenue"].rolling(3).mean().round(2)
monthly_analysis["cumulative"]        = monthly_analysis["revenue"].cumsum().round(2)
monthly_analysis["prev_month"]        = monthly_analysis["revenue"].shift(1)
monthly_analysis["mom_growth_pct"]    = (
    (monthly_analysis["revenue"] / monthly_analysis["prev_month"] - 1) * 100
).round(2)

print(monthly_analysis.tail(8))

      month     revenue  rolling_3m_avg    cumulative  prev_month  \
28  2024-05  6784684.45      5895049.96  1.598531e+08  5420537.23   
29  2024-06  5682686.41      5962636.03  1.655358e+08  6784684.45   
30  2024-07  4582397.80      5683256.22  1.701182e+08  5682686.41   
31  2024-08  5184068.11      5149717.44  1.753022e+08  4582397.80   
32  2024-09  5108498.64      4958321.52  1.804107e+08  5184068.11   
33  2024-10  5841810.73      5378125.83  1.862525e+08  5108498.64   
34  2024-11  5311784.71      5420698.03  1.915643e+08  5841810.73   
35  2024-12  5847564.90      5667053.45  1.974119e+08  5311784.71   

    mom_growth_pct  
28           25.17  
29          -16.24  
30          -19.36  
31           13.13  
32           -1.46  
33           14.35  
34           -9.07  
35           10.09  


In [194]:
# Task 1
# Using monthly_analysis DataFrame above:
# Add a 6-month rolling average column
# called "rolling_6m_avg"
# Print the last 8 rows showing
# month, revenue, rolling_3m_avg, rolling_6m_avg
monthly = (
    orders_clean
    .groupby(orders_clean["order_date"].dt.to_period("M"))["amount"]
    .sum()
    .reset_index()
)
monthly.columns = ["month", "revenue"]
monthly = monthly.sort_values("month")

# 3-month rolling average
monthly["rolling_3m_avg"] = monthly["revenue"].rolling(window = 3).mean().round(2)
monthly["rolling_6m_avg"] = monthly["revenue"].rolling(window = 6).mean().round(2)

print(monthly[["month", "revenue", "rolling_3m_avg", "rolling_6m_avg"]].tail(8))

      month     revenue  rolling_3m_avg  rolling_6m_avg
28  2024-05  6784684.45      5895049.96      5554730.40
29  2024-06  5682686.41      5962636.03      5546765.80
30  2024-07  4582397.80      5683256.22      5498990.51
31  2024-08  5184068.11      5149717.44      5522383.70
32  2024-09  5108498.64      4958321.52      5460478.77
33  2024-10  5841810.73      5378125.83      5530691.02
34  2024-11  5311784.71      5420698.03      5285207.73
35  2024-12  5847564.90      5667053.45      5312687.48


In [197]:
# Task 2
# Using orders_left_full from Section 3.10:
# Rank orders by amount within each country
# Use dense ranking, rank 1 = highest amount
# Add column called "amount_rank_in_country"
# Show top 3 orders per country
# Print showing order_id, country, amount, rank

# Rank customers by total_spent within each country
orders_left_full["amount_rank_in_country"] = (
    orders_left_full
    .groupby("country")["amount"]
    .rank(method="dense", ascending=False)
)

# Show top 3 per country
top_per_country = orders_left_full[
    orders_left_full["amount_rank_in_country"] <= 3
][["order_id", "country", "amount", "amount_rank_in_country"]].sort_values(["country", "amount_rank_in_country"])

print(top_per_country)


     order_id    country    amount  amount_rank_in_country
1365    11375      India  79942.60                     1.0
2450    12467      India  79846.59                     2.0
2144    12159      India  79742.12                     3.0
3999    14026  Singapore  79873.90                     1.0
686     10691  Singapore  79847.49                     2.0
2955    12975  Singapore  79745.98                     3.0
880     10886        UAE  79913.01                     1.0
4672    14704        UAE  79875.26                     2.0
2590    12608        UAE  79849.30                     3.0
1694    11706         UK  79886.60                     1.0
274     10276         UK  79872.73                     2.0
1811    11824         UK  79743.16                     3.0
4604    14635        USA  79811.84                     1.0
402     10405        USA  79629.67                     2.0
4173    14202        USA  79603.04                     3.0


/tmp/ipykernel_6994/2872025398.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("country")["amount"]


In [199]:
# Task 3
# Using monthly_analysis:
# Which month had the highest MoM growth percentage?
# Which month had the lowest (biggest drop)?
# Hint — sort by mom_growth_pct

# Complete monthly analysis in one chain
monthly_analysis = (
    orders_clean
    .groupby(orders_clean["order_date"].dt.to_period("M"))["amount"]
    .sum()
    .reset_index()
)

monthly_analysis.columns = ["month", "revenue"]
monthly_analysis = monthly_analysis.sort_values("month")

monthly_analysis["rolling_3m_avg"]   = monthly_analysis["revenue"].rolling(3).mean().round(2)
monthly_analysis["cumulative"]        = monthly_analysis["revenue"].cumsum().round(2)
monthly_analysis["prev_month"]        = monthly_analysis["revenue"].shift(1)
monthly_analysis["mom_growth_pct"]    = (
    (monthly_analysis["revenue"] / monthly_analysis["prev_month"] - 1) * 100
).round(2)
monthly_analysis = monthly_analysis.sort_values("mom_growth_pct")
# Highest MoM growth
print("Highest growth:")
print(monthly_analysis.sort_values(
    "mom_growth_pct", ascending=False
)[["month", "revenue", "mom_growth_pct"]].head(1))

# Lowest MoM growth
print("\nBiggest drop:")
print(monthly_analysis.sort_values(
    "mom_growth_pct", ascending=True
)[["month", "revenue", "mom_growth_pct"]].head(1))

Highest growth:
     month    revenue  mom_growth_pct
4  2022-05  6746431.8           26.58

Biggest drop:
      month    revenue  mom_growth_pct
30  2024-07  4582397.8          -19.36


**Section 3.12 — Performance + Output + Validation + Method Chaining**

**Four things in one section. Here is exactly what each covers:**

**1. Memory optimisation**
How to reduce DataFrame memory usage — sometimes by 50-70%. The main technique is converting object columns to category dtype. A status column with 5 unique values across 1 million rows stores far less as category than as object.

In [200]:
# Check memory before
print(df.memory_usage(deep=True).sum())

# Convert to category
df["status"] = df["status"].astype("category")

# Check memory after
print(df.memory_usage(deep=True).sum())

1558738
1277377


**2. Output — writing files**

How to write your cleaned DataFrame to different formats:

In [201]:
# CSV — for humans and Excel users
df.to_csv("output.csv", index=False)

# JSON — for APIs
df.to_json("output.json", orient="records")

# Parquet — for pipelines  ← most important
df.to_parquet("output.parquet", index=False)

# Excel — for stakeholders
df.to_excel("output.xlsx", index=False)

**3. Data validation — assertions**

Before writing output always validate your data:

In [202]:
# Assert no nulls in critical columns
assert orders_clean["order_id"].isnull().sum() == 0
assert orders_clean["amount"].isnull().sum() == 0

# Assert valid status values only
valid = {"pending","shipped","delivered","cancelled","returned"}
assert orders_clean["status"].isin(valid).all()

# Assert row count is reasonable
assert len(orders_clean) > 0
assert len(orders_clean) <= len(orders)

**4. Method chaining with pipe()**

Clean professional code chains multiple operations together:

In [ ]:
def add_gst(df):
    df["amount_with_gst"] = df["amount"] * 1.18
    return df

def flag_high_value(df):
    df["is_high_value"] = df["amount"] > 40000
    return df

# Chain everything cleanly
result = (
    orders_clean
    .pipe(add_gst)
    .pipe(flag_high_value)
    .query("status == 'delivered'")
    .groupby("warehouse")["amount_with_gst"]
    .sum()
    .reset_index()
    .sort_values("amount_with_gst", ascending=False)
)

print(result)

Task 1 — Check memory before and after
         converting columns to category

In [210]:
# Check memory before
print("Before:")
print(orders_clean.memory_usage(deep=True))
print(f"Total: {orders_clean.memory_usage(deep=True).sum():,} bytes")

# Convert object columns to category
orders_clean["status"]          = orders_clean["status"].astype("category")
orders_clean["payment_method"]  = orders_clean["payment_method"].astype("category")
orders_clean["warehouse"]       = orders_clean["warehouse"].astype("category")

# Check memory after
print("\nAfter:")
print(orders_clean.memory_usage(deep=True))
print(f"Total: {orders_clean.memory_usage(deep=True).sum():,} bytes")

Before:
Index                171864
order_id             268164
customer_id          263198
product_id           263198
amount                39728
status               284871
payment_method       283923
order_date            39728
delivery_date         39728
warehouse            274052
year                  39728
month                 39728
weekday              243266
quarter               39728
tax_amount            39728
final_amount          39728
discount_amount       39728
discounted_price      39728
is_high_value        268374
status_label         290231
amount_with_gst       39728
is_premium_order       4966
order_size           269429
warehouse_total       39728
pct_of_warehouse      39728
status_avg_amount     39728
dtype: int64
Total: 3,441,728 bytes

After:
Index                171864
order_id             268164
customer_id          263198
product_id           263198
amount                39728
status                 5423
payment_method         5424
order_date            39

Task 2 — Write orders_clean to three formats
         CSV, Parquet, and JSON
         Check file sizes — notice Parquet is smallest

In [211]:
import os

# Write files
orders_clean.to_csv("output.csv", index=False)
orders_clean.to_json("output.json", orient="records")
orders_clean.to_parquet("output.parquet", index=False)

# Check sizes on disk
csv_size     = os.path.getsize("output.csv")
json_size    = os.path.getsize("output.json")
parquet_size = os.path.getsize("output.parquet")

print(f"CSV     : {csv_size:,} bytes")
print(f"JSON    : {json_size:,} bytes")
print(f"Parquet : {parquet_size:,} bytes")
print(f"\nParquet is {csv_size // parquet_size}x smaller than CSV")

CSV     : 1,066,648 bytes
JSON    : 2,934,314 bytes
Parquet : 359,867 bytes

Parquet is 2x smaller than CSV


Task 3 — Write a pipe() chain that:
         filters delivered orders
         adds amount_with_gst column
         groups by country and sums revenue
         sorts descending

In [215]:
def add_gst(df):
    df = df.copy()
    df["amount_with_gst"] = df["amount"] * 1.18
    return df

result = (
    orders_clean
    .merge(
        customers_clean[["customer_id", "country"]],
        on  = "customer_id",
        how = "left"
    )
    .pipe(add_gst)
    .query("status == 'delivered'")
    .groupby("country",observed=True)["amount_with_gst"]
    .sum()
    .reset_index()
    .sort_values("amount_with_gst", ascending=False)
)
result["amount_with_gst"] = result["amount_with_gst"].round(2)
print(result)

     country  amount_with_gst
4        USA      24539590.47
3         UK      24391135.37
1  Singapore      23266081.39
2        UAE      22750531.79
0      India      21902069.57


1. Reads all 4 TradeSphere CSV files
2. Cleans and validates each one
3. Merges into one analytical dataset
4. Aggregates — revenue by category, country, month
5. Validates output — assertions before writing
6. Writes to Parquet
7. Uses pipe() for clean structure

In [216]:
import pandas as pd
from pathlib import Path

# Base path
DATA_PATH = Path("tradesphere")

def read_data(data_path: Path) -> dict:
    """
    Reads all required CSV files into a dictionary of DataFrames.
    """
    files = {
        "customers": "customers.csv",
        "products": "products.csv",
        "orders": "orders.csv",
        "payments": "payments.csv"
    }

    dfs = {}

    for name, file in files.items():
        file_path = data_path / file
        df = pd.read_csv(file_path)
        dfs[name] = df

    return dfs


# Execute
data = read_data(DATA_PATH)

# Access individual datasets
customers = data["customers"]
products  = data["products"]
orders    = data["orders"]
payments  = data["payments"]

In [217]:
def clean_and_validate_all(data: dict) -> dict:

    # -------------------- CUSTOMERS --------------------
    customers = data["customers"].copy()
    customers.columns = customers.columns.str.strip().str.lower()

    customers["name"] = customers["name"].str.strip()
    customers = customers[customers["name"] != ""]

    customers["status"] = customers["status"].str.lower().str.strip()
    customers = customers[customers["status"].isin(["active", "inactive"])]

    customers["total_spent"] = pd.to_numeric(customers["total_spent"], errors="coerce").fillna(0)
    customers["join_date"] = pd.to_datetime(customers["join_date"], errors="coerce")

    assert customers["customer_id"].notna().all(), "Customer ID nulls"
    assert customers["join_date"].notna().all(), "Invalid join_date"

    # -------------------- PRODUCTS --------------------
    products = data["products"].copy()
    products.columns = products.columns.str.strip().str.lower()

    products["category"] = products["category"].str.strip().replace("", "unknown")

    products["price"] = pd.to_numeric(products["price"], errors="coerce")
    products = products[products["price"].notna()]
    products = products[products["price"] > 0]

    products = products[products["stock_qty"] >= 0]
    products["is_active"] = products["is_active"].astype(bool)

    assert products["product_id"].notna().all(), "Missing product_id"
    assert (products["price"] > 0).all(), "Invalid price"

    # -------------------- ORDERS --------------------
    orders = data["orders"].copy()
    orders.columns = orders.columns.str.strip().str.lower()

    orders["status"] = orders["status"].str.lower().str.strip()
    orders = orders[
        orders["status"].isin(["pending", "shipped", "delivered", "cancelled", "returned"])
    ]

    orders["amount"] = pd.to_numeric(orders["amount"], errors="coerce")
    orders = orders[orders["amount"].notna()]
    orders = orders[orders["amount"] > 0]

    orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
    orders["delivery_date"] = pd.to_datetime(orders["delivery_date"], errors="coerce")

    orders = orders[
        (orders["delivery_date"].isna()) |
        (orders["delivery_date"] >= orders["order_date"])
    ]

    assert orders["order_id"].notna().all(), "Missing order_id"
    assert orders["customer_id"].notna().all(), "Missing customer_id"
    assert orders["product_id"].notna().all(), "Missing product_id"
    assert orders["order_date"].notna().all(), "Invalid order_date"

    # -------------------- PAYMENTS --------------------
    payments = data["payments"].copy()
    payments.columns = payments.columns.str.strip().str.lower()

    payments["status"] = payments["status"].str.lower().str.strip()
    payments = payments[payments["status"].isin(["success", "failed", "refunded"])]

    payments["amount"] = pd.to_numeric(payments["amount"], errors="coerce")
    payments = payments[payments["amount"].notna()]
    payments = payments[payments["amount"] >= 0]

    payments["payment_date"] = pd.to_datetime(payments["payment_date"], errors="coerce")

    assert payments["payment_id"].notna().all(), "Missing payment_id"
    assert payments["order_id"].notna().all(), "Missing order_id"
    assert payments["payment_date"].notna().all(), "Invalid payment_date"

    # -------------------- RETURN --------------------
    return {
        "customers": customers,
        "products": products,
        "orders": orders,
        "payments": payments
    }

In [218]:
def build_analytical_dataset(cleaned: dict) -> pd.DataFrame:

    customers = cleaned["customers"]
    products  = cleaned["products"]
    orders    = cleaned["orders"]
    payments  = cleaned["payments"]

    # -------------------- Merge Customers --------------------
    df = orders.merge(
        customers,
        on="customer_id",
        how="left",
        suffixes=("", "_customer")
    )

    # -------------------- Merge Products --------------------
    df = df.merge(
        products,
        on="product_id",
        how="left",
        suffixes=("", "_product")
    )

    # -------------------- Merge Payments --------------------
    df = df.merge(
        payments,
        on="order_id",
        how="left",
        suffixes=("", "_payment")
    )

    # -------------------- Validation --------------------
    assert df.shape[0] == orders.shape[0], "Row count changed after joins"

    return df

In [219]:
def aggregate_revenue(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    # -------------------- Create Month Column --------------------
    df["order_month"] = df["order_date"].dt.to_period("M").astype(str)

    # -------------------- Aggregate --------------------
    agg_df = (
        df.groupby(["category", "country", "order_month"], as_index=False)
          .agg(
              total_revenue=("amount", "sum"),
              total_orders=("order_id", "nunique")
          )
    )

    # -------------------- Validation --------------------
    assert agg_df["total_revenue"].notna().all(), "Revenue contains nulls"
    assert (agg_df["total_revenue"] >= 0).all(), "Negative revenue found"

    return agg_df

In [220]:
def final_validation_and_write(df: pd.DataFrame, output_path: str):

    # -------------------- Final Validations --------------------

    # No null critical fields
    assert df["category"].notna().all(), "Null category found"
    assert df["country"].notna().all(), "Null country found"
    assert df["order_month"].notna().all(), "Null month found"

    # Revenue sanity
    assert (df["total_revenue"] >= 0).all(), "Negative revenue found"

    # Orders sanity
    assert (df["total_orders"] > 0).all(), "Zero orders found"

    # -------------------- Write to Parquet --------------------
    df.to_parquet(output_path, index=False)

    print(f"✅ Data written to {output_path}")

In [223]:
def pipeline(data_path: Path, output_path: str):
    print("Reading data...")
    raw = read_data(data_path)

    print("Cleaning and validating...")
    cleaned = clean_and_validate_all(raw)

    print("Building analytical dataset...")
    df = build_analytical_dataset(cleaned)

    print("Aggregating revenue...")
    agg = aggregate_revenue(df)

    print("Validating and writing output...")
    final_validation_and_write(agg, output_path)

    print(f"Done — {len(agg):,} rows written")
    return agg

# Run it
result = pipeline(
    data_path   = DATA_PATH,
    output_path = "tradesphere/analytical_output.parquet"
)

print(result.head(10))

Reading data...
Cleaning and validating...
Building analytical dataset...
Aggregating revenue...
Validating and writing output...
✅ Data written to tradesphere/analytical_output.parquet
Done — 896 rows written
  category country order_month  total_revenue  total_orders
0    Books   India     2022-01      195534.78             4
1    Books   India     2022-02      258559.51             7
2    Books   India     2022-03      144336.04             2
3    Books   India     2022-04       27783.65             3
4    Books   India     2022-05      151811.84             3
5    Books   India     2022-06      271443.26             7
6    Books   India     2022-07       66716.24             1
7    Books   India     2022-08      190229.86             6
8    Books   India     2022-09       74542.00             2
9    Books   India     2022-10      273833.99             6
